In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/policies_contents.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/test_contents.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/claims.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/policies_buildings.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/weather.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/sample_submission.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/customers.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/addresses.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/test_buildings.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed/test_motor.parquet
/kaggle/input/competitions/iaa-ai-assist

Phase 1

In [2]:
# =============================================================================
# PART 2 - PHASE 1: LOAD, CLEAN EXPOSURE, MATCH CLAIMS, JOIN
# Runs BOTH markets in one go: every cell loops over `markets`, so each cell's
# output shows the developed results, then the emerging results.
#
# Python port of the Week 1 R pipeline (Kaggle no longer runs R). Same logic
# and checks; the developed counts are reconciled with the Week 1 R run in
# Cell 9 before anything is built on them.
#
# Markets are never joined: each market's tables are read from its own folder
# and joined only to each other. Results are kept per market (P1[market]).
# No years, row counts or dataset-specific values are hard-coded.
#
# Output (in /kaggle/working/prep_<market>):
#   train_<line>.parquet, test_<line>.parquet   joined policy rows (one per row and
#                                               calendar year; orig_row_id = Kaggle row_id)
#   claims_matched.parquet                      claims with the row_id they belong to
#   claims_double_matched.parquet               claims that fell in two rows (resolved)
#   claims_unmatched.parquet                    claims outside every row (excluded)
#   phase1_summary.csv, session_info.txt
# =============================================================================

# %% Phase 1 Cell 1: Settings and packages =====================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
markets         = ["developed", "emerging"]   # folder names inside the competition data
data_dirs       = {}             # e.g. {"emerging": "/kaggle/input/.../emerging"}; empty = find automatically
lines           = ["motor", "buildings", "contents"]
days_per_year   = 365            # data convention: every policy year counts as 365 days
leap_ends_dec30 = True           # data convention: in leap years cover stops on 30 Dec
drop_cols       = ["renewal_decision",                         # training only: leakage
                   "surname", "email_domain", "branch_id",     # admin / marketing fields
                   "campaign_code", "marketing_segment", "document_preference"]
# ---------------------------------------------------------------------------

import os, glob, sys, platform, warnings
from datetime import datetime
import numpy as np
import pandas as pd

work_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def find_data_dir(market):
    """The folder named after the market that contains customers.parquet."""
    hits = sorted({os.path.dirname(p) for root in ["/kaggle/input", "data"]
                   for p in glob.glob(os.path.join(root, "**", "customers.parquet"), recursive=True)})
    hits = [h for h in hits if os.path.basename(os.path.normpath(h)) == market]
    if len(hits) != 1:
        raise RuntimeError(f"Found {len(hits)} folders for market '{market}': {hits}. Set data_dirs in Cell 1.")
    return hits[0]

P1 = {}   # everything Phase 1 builds, per market
for m in markets:
    d_in  = data_dirs.get(m) or find_data_dir(m)
    d_out = os.path.join(work_dir, f"prep_{m}")
    os.makedirs(d_out, exist_ok=True)
    P1[m] = {"data_dir": d_in, "out_dir": d_out}
    print(f"{m:<10} raw data: {d_in}\n{'':<10} output:   {d_out}")

developed  raw data: /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed
           output:   /kaggle/working/prep_developed
emerging   raw data: /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/emerging
           output:   /kaggle/working/prep_emerging


In [3]:
# %% Phase 1 Cell 2: Helper functions ==========================================

id_cols   = ["row_id", "policy_id", "customer_id", "address_id", "claim_id",
             "repairer_id", "third_party_id"]
date_cols = ["period_start", "period_end", "loss_date", "report_date", "date_of_birth"]

def as_id(s):
    """IDs as text. Whole numbers stored as numbers lose any '.0'; blanks stay blank."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype("Int64").astype("string")
    return s.astype("string")

def read_raw(market, name):
    """Read one raw file of a market: IDs as text, dates as dates, other text lowercased and trimmed."""
    folder = P1[market]["data_dir"]
    path, csv = os.path.join(folder, name + ".parquet"), os.path.join(folder, name + ".csv")
    if os.path.exists(path):  df = pd.read_parquet(path)
    elif os.path.exists(csv): df = pd.read_csv(csv, dtype=str)
    else: raise FileNotFoundError(f"Missing file: {path}")
    for c in df.columns:
        if c in id_cols:
            df[c] = as_id(df[c])
        elif c in date_cols:
            df[c] = pd.to_datetime(df[c])
        elif pd.api.types.is_string_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
            vals = df[c].dropna()
            if len(vals) and vals.map(lambda v: isinstance(v, str)).all():
                df[c] = df[c].str.strip().str.lower()
    return df

def read_raw_optional(market, name):
    """Optional file: None if it isn't in this market's folder."""
    folder = P1[market]["data_dir"]
    exists = any(os.path.exists(os.path.join(folder, name + ext)) for ext in (".parquet", ".csv"))
    return read_raw(market, name) if exists else None

def repair_ids(x, ref, label):
    """Restore leading zeros lost when a file stored IDs as numbers. Pads only all-digit
    IDs shorter than the reference IDs, and only when the reference IDs are all digits
    of a single width - otherwise returns x unchanged."""
    ref = pd.Series(ref).dropna().unique()
    widths = {len(r) for r in ref}
    if len(widths) != 1 or not all(str(r).isdigit() for r in ref):
        return x
    w = widths.pop()
    short = x.notna() & x.str.fullmatch(r"\d+").fillna(False) & (x.str.len() < w)
    if short.any():
        print(f"  Repaired {int(short.sum())} {label} values missing leading zeros")
    x = x.copy()
    x[short] = x[short].str.zfill(w)
    return x

def check_unique(df, keys, label):
    """Stop if a lookup table has duplicate keys (a join would multiply rows)."""
    dups = df.duplicated(subset=keys, keep=False)
    if dups.any():
        print(df.loc[dups, keys].head())
        raise RuntimeError(f"{label} has duplicated keys on {' + '.join(keys)}")
    print(f"  OK: {label} is unique on {' + '.join(keys)}")

def year_last_day(y):
    """Last covered day of each calendar year under the data's convention."""
    y = pd.Series(y)
    leap = (y % 4 == 0) & ((y % 100 != 0) | (y % 400 == 0))
    day = np.where(leap_ends_dec30 & leap, 30, 31)
    return pd.to_datetime(pd.DataFrame({"year": y, "month": 12, "day": day}))

def clean_policies(df, name):
    """Policy rows -> exposure in years. Works for any years:
      1. A row crossing a year end is split into one piece per calendar year.
         Pieces get row_id '<row_id>_<year>'; orig_row_id keeps the original
         (predictions are summed back to orig_row_id for submission).
      2. Each piece is cut at the year's last covered day (30 Dec in leap years).
      3. Days already covered by an earlier piece of the same policy are removed.
      4. Exposure = days / days_per_year, then hard checks that nothing exceeds 1."""
    if (df["period_end"] < df["period_start"]).any():
        raise RuntimeError(f"{name}: rows end before they start")
    n_in = len(df)
    df = df.copy()
    df["orig_row_id"] = df["row_id"]
    df["exposure_kaggle"] = df["exposure"]
    y0, y1 = df["period_start"].dt.year, df["period_end"].dt.year
    df["_n"] = (y1 - y0 + 1).astype(int)
    df["_row_order"] = np.arange(n_in)
    df = df.loc[df.index.repeat(df["_n"])].copy()
    df["_k"] = df.groupby("_row_order").cumcount()
    df["_y"] = df["period_start"].dt.year + df["_k"]
    df["_split"] = df["_n"] > 1
    yr_start = pd.to_datetime(pd.DataFrame({"year": df["_y"], "month": 1, "day": 1}))
    yr_end   = pd.to_datetime(pd.DataFrame({"year": df["_y"], "month": 12, "day": 31}))
    df["period_start"] = np.where(df["_k"] == 0, df["period_start"], yr_start)
    df["period_end"]   = np.where(df["_y"] == df["period_end"].dt.year, df["period_end"], yr_end)
    last = year_last_day(df["_y"].to_numpy()).to_numpy()
    df["_trimmed"] = df["period_end"].to_numpy() > last
    df["period_end"] = np.minimum(df["period_end"].to_numpy(), last)
    df["row_id"] = np.where(df["_split"], df["orig_row_id"] + "_" + df["_y"].astype(str), df["orig_row_id"])
    df.loc[df["_split"], "exposure_kaggle"] = np.nan
    n_split, n_trimmed = df.loc[df["_split"], "orig_row_id"].nunique(), int(df["_trimmed"].sum())

    # remove days already covered by an earlier row of the same policy
    df["_piece"] = np.arange(len(df))
    df = df.sort_values(["policy_id", "period_start", "period_end"], kind="mergesort")
    start = df["period_start"].values.astype("datetime64[D]").astype(np.int64)
    end   = df["period_end"].values.astype("datetime64[D]").astype(np.int64)
    df["_end"] = end
    prev_end = df.groupby("policy_id")["_end"].cummax().groupby(df["policy_id"]).shift(1).to_numpy()
    eff_start = np.where(np.isnan(prev_end), start, np.maximum(start, np.nan_to_num(prev_end) + 1))
    df["_overlap"] = eff_start - start
    df["_days"] = np.maximum(0, end - eff_start + 1)
    df["exposure"] = df["_days"] / days_per_year
    df = df.sort_values("_piece")

    # hard checks: exposure can never exceed 1
    if (df["exposure"] > 1).any():
        raise RuntimeError(f"{name}: a row exceeds 1 year - check the settings in Cell 1")
    per_year = df.groupby([df["policy_id"], df["period_start"].dt.year]).agg(t=("exposure", "sum"), d=("_days", "sum"))
    if (per_year["t"] > 1 + 1e-9).any():
        raise RuntimeError(f"{name}: a policy exceeds 1 year of exposure in a calendar year")

    # convention check: full policy-years should cover days_per_year days
    near_full = per_year["d"][per_year["d"] >= days_per_year - 5]
    typical = int(near_full.mode().iloc[0]) if len(near_full) else None
    if typical is not None and typical != days_per_year:
        warnings.warn(f"{name}: full policy-years usually cover {typical} days, not {days_per_year}"
                      " - the date convention may differ; review the settings in Cell 1")

    print(f"  {name:<26} rows in: {n_in:>9,} | split across years: {n_split:>5} | cut at leap-year 30 Dec: {n_trimmed:>5}"
          f" | overlap days removed on: {int((df['_overlap'] > 0).sum()):>5} | zero exposure: {int((df['exposure'] == 0).sum()):>4}"
          f" | full year = {typical} days | max: {df['exposure'].max():.4f}")
    n_diff = int(((df["exposure"] - df["exposure_kaggle"]).abs() > 0.01).sum())
    if n_diff:
        print(f"    exposure differs from Kaggle's by >0.01 on {n_diff} unsplit rows")
    if (df["exposure"] == 0).any():
        warnings.warn(f"{name}: {int((df['exposure'] == 0).sum())} rows have zero exposure - inspect before modelling")
    return df.drop(columns=[c for c in df.columns if c.startswith("_")] + [c for c in drop_cols if c in df.columns]).reset_index(drop=True)

In [4]:
# %% Phase 1 Cell 3: Load reference tables and claims ==========================

for m in markets:
    r = P1[m]
    r["customers"]  = read_raw(m, "customers")
    r["addresses"]  = read_raw(m, "addresses")
    r["weather"]    = read_raw(m, "weather")
    r["claims"]     = read_raw(m, "claims")
    r["sample_sub"] = read_raw_optional(m, "sample_submission")   # only the market being predicted has one
    r["weather"]["address_id"] = repair_ids(r["weather"]["address_id"], r["addresses"]["address_id"], f"{m} weather address_id")
    print(f"{m:<10} rows - customers: {len(r['customers']):,} | addresses: {len(r['addresses']):,} | "
          f"weather: {len(r['weather']):,} | claims: {len(r['claims']):,} | sample_submission: "
          + ("none (not predicted)" if r["sample_sub"] is None else f"{len(r['sample_sub']):,}"))

developed  rows - customers: 545,240 | addresses: 575,983 | weather: 2,879,915 | claims: 163,150 | sample_submission: 626,639
emerging   rows - customers: 135,856 | addresses: 136,914 | weather: 410,742 | claims: 9,258 | sample_submission: 209,063


In [5]:
# %% Phase 1 Cell 4: Load and clean policy files (always from the raw files) ===

for m in markets:
    r = P1[m]
    print(f"\n{m}:")
    def load_policies(file, name):
        df = read_raw(m, file)
        df["customer_id"] = repair_ids(df["customer_id"], r["customers"]["customer_id"], f"{name} customer_id")
        df["address_id"]  = repair_ids(df["address_id"],  r["addresses"]["address_id"],  f"{name} address_id")
        return clean_policies(df, name)
    r["train"] = {l: load_policies(f"policies_{l}", f"{m} train {l}") for l in lines}
    r["test"]  = {l: load_policies(f"test_{l}",     f"{m} test {l}")  for l in lines}
    all_policy_ids = pd.concat([d["policy_id"] for d in list(r["train"].values()) + list(r["test"].values())])
    r["claims"]["policy_id"] = repair_ids(r["claims"]["policy_id"], all_policy_ids, f"{m} claims policy_id")


developed:
  Repaired 98110 developed train motor customer_id values missing leading zeros
  developed train motor      rows in:   979,968 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 4474 unsplit rows
  developed train buildings  rows in:   707,726 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 3365 unsplit rows
  developed train contents   rows in:   721,617 | split across years:     0 | cut at leap-year 30 Dec:     0 | overlap days removed on:     0 | zero exposure:    0 | full year = 365 days | max: 1.0000
    exposure differs from Kaggle's by >0.01 on 3434 unsplit rows
  Repaired 25554 developed test motor customer_id values missing leading zeros
  developed test motor       ro

In [6]:
# %% Phase 1 Cell 5: Key checks ================================================

for m in markets:
    r = P1[m]
    print(f"\n{m}:")
    check_unique(r["customers"], ["customer_id"], "customers")
    check_unique(r["addresses"], ["address_id"], "addresses")
    check_unique(r["weather"], ["address_id", "year"], "weather")
    check_unique(r["claims"], ["claim_id"], "claims")
    for l in lines:
        check_unique(r["train"][l], ["row_id"], f"train {l}")
        check_unique(r["test"][l],  ["row_id"], f"test {l}")

    odd_lines = set(r["claims"]["line"].dropna()) - set(lines)
    if odd_lines:
        warnings.warn(f"{m}: unexpected claim line values: {sorted(odd_lines)}")

    pid_lines = pd.concat([pd.DataFrame({"policy_id": r["train"][l]["policy_id"].unique(), "line": l}) for l in lines])
    print("  policy_ids in more than one line:", int((pid_lines["policy_id"].value_counts() > 1).sum()))

    test_ids = pd.concat([d["orig_row_id"] for d in r["test"].values()]).unique()
    ss = r["sample_sub"]
    if ss is None:
        print(f"  No sample_submission for {m} - test rows not checked (this market is not predicted)")
    elif set(test_ids) != set(ss["row_id"]) or len(test_ids) != len(ss):
        raise RuntimeError(f"{m}: test row_ids don't match sample_submission ({len(test_ids)} vs {len(ss)})")
    else:
        print(f"  OK: all {len(test_ids):,} test row_ids match sample_submission")


developed:
  OK: customers is unique on customer_id
  OK: addresses is unique on address_id
  OK: weather is unique on address_id + year
  OK: claims is unique on claim_id
  OK: train motor is unique on row_id
  OK: test motor is unique on row_id
  OK: train buildings is unique on row_id
  OK: test buildings is unique on row_id
  OK: train contents is unique on row_id
  OK: test contents is unique on row_id
  policy_ids in more than one line: 0
  OK: all 626,639 test row_ids match sample_submission

emerging:
  OK: customers is unique on customer_id
  OK: addresses is unique on address_id
  OK: weather is unique on address_id + year
  OK: claims is unique on claim_id
  OK: train motor is unique on row_id
  OK: test motor is unique on row_id
  OK: train buildings is unique on row_id
  OK: test buildings is unique on row_id
  OK: train contents is unique on row_id
  OK: test contents is unique on row_id
  policy_ids in more than one line: 0
  OK: all 209,063 test row_ids match sample_su

In [7]:
# %% Phase 1 Cell 6: Match claims to policy rows ===============================
# A claim belongs to the row of its policy whose period contains its loss_date.
# If two rows qualify (shared changeover day), the earlier row is used - the
# same row that keeps that day's exposure in clean_policies(), so claims and
# exposure stay consistent. Claims outside every row are set aside, with the reason.

def match_claims(claims, train_line, line_name, label):
    cl  = claims[claims["line"] == line_name]
    pol = train_line[["row_id", "orig_row_id", "policy_id", "period_start", "period_end"]]
    mt = cl.merge(pol, on="policy_id", how="inner")
    mt = mt[(mt["loss_date"] >= mt["period_start"]) & (mt["loss_date"] <= mt["period_end"])]
    n_rows   = mt.groupby("claim_id")["row_id"].transform("size")
    doubles  = mt[n_rows > 1].assign(n_rows=n_rows[n_rows > 1])
    resolved = mt.sort_values(["claim_id", "period_start"], kind="mergesort").drop_duplicates("claim_id")

    ranges = pol.groupby("policy_id").agg(first_start=("period_start", "min"), last_end=("period_end", "max")).reset_index()
    unmatched = cl[~cl["claim_id"].isin(resolved["claim_id"])].merge(ranges, on="policy_id", how="left")
    unmatched["reason"] = np.select(
        [unmatched["first_start"].isna(), unmatched["loss_date"] < unmatched["first_start"],
         unmatched["loss_date"] > unmatched["last_end"]],
        ["policy_id not in policy file", "before first row", "after last row"], "in a gap between rows")
    unmatched = unmatched.drop(columns=["first_start", "last_end"])

    print(f"  {label:<20} claims: {len(cl):>8,} | matched: {len(resolved):>8,} | double-matched: "
          f"{doubles['claim_id'].nunique():>5} | unmatched: {len(unmatched):>5}")
    if len(unmatched):
        print(unmatched.groupby(["reason", "status"]).size().rename("n").reset_index().to_string(index=False))
    return {"matched": resolved, "doubles": doubles, "unmatched": unmatched}

for m in markets:
    r = P1[m]
    print(f"\n{m}:")
    r["claim_results"] = {l: match_claims(r["claims"], r["train"][l], l, f"{m} {l}") for l in lines}
    r["claims_matched"]   = pd.concat([x["matched"] for x in r["claim_results"].values()], ignore_index=True)
    r["claims_doubles"]   = pd.concat([x["doubles"] for x in r["claim_results"].values()], ignore_index=True)
    r["claims_unmatched"] = pd.concat([x["unmatched"] for x in r["claim_results"].values()], ignore_index=True)
    assert not r["claims_matched"]["claim_id"].duplicated().any()


developed:
  developed motor      claims:  112,606 | matched:  112,309 | double-matched:     0 | unmatched:   297
          reason     status   n
before first row     closed 283
before first row       open  11
before first row repudiated   3
  developed buildings  claims:   26,442 | matched:   26,375 | double-matched:     0 | unmatched:    67
          reason status  n
before first row closed 64
before first row   open  3
  developed contents   claims:   24,102 | matched:   24,052 | double-matched:     0 | unmatched:    50
          reason     status  n
before first row     closed 48
before first row       open  1
before first row repudiated  1

emerging:
  emerging motor       claims:    7,183 | matched:    7,155 | double-matched:     0 | unmatched:    28
          reason status  n
before first row closed 27
before first row   open  1
  emerging buildings   claims:    1,059 | matched:    1,056 | double-matched:     0 | unmatched:     3
          reason status  n
before first row clos

In [8]:
# %% Phase 1 Cell 7: Join customers, addresses and weather (within each market)
# Weather joins on address and the calendar year of period_start. If that year
# is missing, the address's latest earlier year is used (weather_imputed = True).

def add_weather(pol, weather):
    weather = weather.assign(year=weather["year"].astype("int64"))
    wcols = [c for c in weather.columns if c not in ("address_id", "year")]
    pol = pol.assign(weather_year=pol["period_start"].dt.year.astype("int64"))
    out = pol.merge(weather, how="left", left_on=["address_id", "weather_year"], right_on=["address_id", "year"]).drop(columns="year")
    out["weather_imputed"] = out[wcols[0]].isna()
    if out["weather_imputed"].any():
        need = out.loc[out["weather_imputed"], ["row_id", "address_id", "weather_year"]].sort_values("weather_year")
        fb = pd.merge_asof(need, weather.sort_values("year"), left_on="weather_year", right_on="year",
                           by="address_id", direction="backward").set_index("row_id")[wcols]
        idx = out["weather_imputed"]
        out.loc[idx, wcols] = fb.loc[out.loc[idx, "row_id"]].to_numpy()
    return out, wcols

def build_features(pol, r, name):
    out = (pol.merge(r["customers"].assign(_cust=True), on="customer_id", how="left")
              .merge(r["addresses"].assign(_addr=True), on="address_id", how="left"))
    out, wcols = add_weather(out, r["weather"])
    print(f"  {name:<26} no customer: {int(out['_cust'].isna().sum()):>6} | no address: {int(out['_addr'].isna().sum()):>6}"
          f" | weather filled from earlier year: {int(out['weather_imputed'].sum()):>6}"
          f" | weather still missing: {int(out[wcols[0]].isna().sum()):>6}")
    out = out.drop(columns=["_cust", "_addr"] + [c for c in drop_cols if c in out.columns])
    assert len(out) == len(pol) and not out["row_id"].duplicated().any()   # joins never add or drop rows
    return out

for m in markets:
    r = P1[m]
    print(f"\n{m}:")
    r["train_full"] = {l: build_features(r["train"][l], r, f"{m} train {l}") for l in lines}
    r["test_full"]  = {l: build_features(r["test"][l],  r, f"{m} test {l}")  for l in lines}
    for l in lines:
        diff_cols = set(r["train_full"][l].columns) ^ set(r["test_full"][l].columns)
        if diff_cols: warnings.warn(f"{m} {l}: train/test column mismatch: {sorted(diff_cols)}")
        else: print(f"  OK: {l} train and test have the same columns")


developed:
  developed train motor      no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  developed train buildings  no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  developed train contents   no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  developed test motor       no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  developed test buildings   no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  developed test contents    no customer:      0 | no address:      0 | weather filled from earlier year:      0 | weather still missing:      0
  OK: motor train and test have the same columns
  OK: buildings train and test have the same columns
  OK: contents t

In [9]:
# %% Phase 1 Cell 8: Save outputs, summary and session info ====================

summary_parts = []
for m in markets:
    r, out = P1[m], P1[m]["out_dir"]
    for l in lines:
        r["train_full"][l].to_parquet(os.path.join(out, f"train_{l}.parquet"), index=False)
        r["test_full"][l].to_parquet(os.path.join(out, f"test_{l}.parquet"), index=False)
    r["claims_matched"].to_parquet(os.path.join(out, "claims_matched.parquet"), index=False)
    r["claims_doubles"].to_parquet(os.path.join(out, "claims_double_matched.parquet"), index=False)
    r["claims_unmatched"].to_parquet(os.path.join(out, "claims_unmatched.parquet"), index=False)

    s = pd.DataFrame([{
        "market":           m,
        "line":             l,
        "train_rows":       r["train_full"][l]["orig_row_id"].nunique(),
        "train_pieces":     len(r["train_full"][l]),
        "test_rows":        r["test_full"][l]["orig_row_id"].nunique(),
        "test_pieces":      len(r["test_full"][l]),
        "train_exposure":   round(r["train_full"][l]["exposure"].sum(), 2),
        "claims":           int((r["claims"]["line"] == l).sum()),
        "claims_matched":   len(r["claim_results"][l]["matched"]),
        "claims_double":    r["claim_results"][l]["doubles"]["claim_id"].nunique(),
        "claims_unmatched": len(r["claim_results"][l]["unmatched"]),
    } for l in lines])
    s.to_csv(os.path.join(out, "phase1_summary.csv"), index=False)
    with open(os.path.join(out, "session_info.txt"), "w") as fh:
        fh.write(f"Run: {datetime.now():%Y-%m-%d %H:%M}\nMarket: {m}\nData: {r['data_dir']}\n"
                 f"Python: {sys.version}\nPlatform: {platform.platform()}\n"
                 f"pandas: {pd.__version__}\nnumpy: {np.__version__}\n")
    r["summary"] = s
    summary_parts.append(s)

summary_tbl = pd.concat(summary_parts, ignore_index=True)
print(summary_tbl.to_string(index=False))
for m in markets:
    print(f"\nPhase 1 complete for {m}. Files in {P1[m]['out_dir']}:", sorted(os.listdir(P1[m]["out_dir"])))

   market      line  train_rows  train_pieces  test_rows  test_pieces  train_exposure  claims  claims_matched  claims_double  claims_unmatched
developed     motor      979968        979968     254986       254986       787157.52  112606          112309              0               297
developed buildings      707726        707726     184155       184155       590560.18   26442           26375              0                67
developed  contents      721617        721617     187498       187498       590859.28   24102           24052              0                50
 emerging     motor       47283         47283      90521        90521        38968.53    7183            7155              0                28
 emerging buildings       27447         27447      52295        52295        23468.01    1059            1056              0                 3
 emerging  contents       34566         34566      66247        66247        29057.98    1016            1011              0                 5

In [10]:
# %% Phase 1 Cell 9: Reconcile with the Week 1 R run, then free memory =========
# The developed folder is the Part 1 data, so the Python port must reproduce
# the counts the Week 1 R pipeline recorded. Any difference stops the run.
# Reference values come from the Week 1 R notebook (Version 6) and its logs.

week1_r = {"claims_matched": 162736, "claims_unmatched": 414, "test_rows": 626639}

if "developed" in P1:
    s = P1["developed"]["summary"]
    py = {"claims_matched": int(s["claims_matched"].sum()),
          "claims_unmatched": int(s["claims_unmatched"].sum()),
          "test_rows": int(s["test_rows"].sum())}
    recon = pd.DataFrame({"week1_R": week1_r, "python": py})
    recon["match"] = recon["week1_R"] == recon["python"]
    print(recon.to_string())
    if not recon["match"].all():
        raise RuntimeError("Python port does not reproduce the Week 1 R counts - investigate before continuing")
    print("OK: Python port reproduces the Week 1 R counts")
else:
    print("Developed market not run - no reconciliation with Week 1.")

# Phase 2 reads the saved files, so the large in-memory tables can go
P1 = {m: {"data_dir": P1[m]["data_dir"], "out_dir": P1[m]["out_dir"]} for m in P1}
import gc; gc.collect()
print("Phase 1 tables cleared from memory (all outputs are saved to disk).")

                  week1_R  python  match
claims_matched     162736  162736   True
claims_unmatched      414     414   True
test_rows          626639  626639   True
OK: Python port reproduces the Week 1 R counts
Phase 1 tables cleared from memory (all outputs are saved to disk).


Phase 2

In [11]:
# =============================================================================
# PART 2 - PHASE 2: CLEANING  (Python port of the Part 1 R Phase 2, plus the
# emerging-market fixes)
#
# Runs BOTH markets in one go: each cell loops over `markets`. For each market
# it reads the joined tables Phase 1 saved, cleans train and test together with
# the same rules, and saves model-ready tables. Nothing depends
# on particular years or values: fill values come from each run's training
# data, and any new undocumented code stops the run with its name.
#
# Rules (evidence in the Phase 2 log):
#   - Text lowercased/trimmed, spaces and hyphens -> "_"; known codes mapped:
#       developed t, w -> timber (2023 system change, confirmed by the host)
#       emerging BRK/STN/WOD/MIX -> brick/stone/timber/non_standard (documented)
#   - Any other undocumented category value -> run stops and lists it
#   - sum_insured entered 1,000x too large -> divided by 1,000, flagged
#   - occupation free text: "self-employed" prefix split into a flag
#   - Blank text and yes/no values -> "unknown"; blank numbers -> training
#     median + <column>_missing flag
#   - telematics_score blank when not opted in -> opted-in training median,
#     with telematics_score_known flag (developed only)
#   - years_licensed capped at the oldest age the driver can be minus
#     min_licence_age; ncb_years capped at years_licensed; both flagged
#   - Both markets get the same age bands (developed ages banded to the
#     emerging bands) so the markets can share one model
#   - Derived: policy_year, customer_age, customer_tenure, property_age
#
# Output (in /kaggle/working/clean_<market>): train_<line>.parquet,
# test_<line>.parquet, cleaning_log.csv, fill_values.csv
# =============================================================================

# %% Phase 2 Cell 1: Settings ==================================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
markets         = ["developed", "emerging"]
in_prefix       = "prep_"                 # Phase 1 output folders: prep_<market>
out_prefix      = "clean_"                # Phase 2 output folders: clean_<market>
lines           = ["motor", "buildings", "contents"]
unknown_label   = "unknown"
min_licence_age = 16    # nobody holds a licence before this age

# Undocumented or local codes and the documented value they stand for.
code_map = {
    "construction_type": {
        "t": "timber", "w": "timber",              # developed: coding change in 2023
        "brk": "brick", "stn": "stone",            # emerging local codes (data description)
        "wod": "timber", "mix": "non_standard",
    },
}

# Blanks that mean something specific (the description says "if any")
na_meaning = {"campaign_code": "none"}

# Age bands used in the emerging market; developed ages are banded the same way
age_bands = [(17, 24, "17_24"), (25, 34, "25_34"), (35, 49, "35_49"), (50, 64, "50_64"), (65, 200, "65+")]

# sum_insured unit errors: a value is treated as entered 1,000x too large if it is
# more than si_ratio x the line's training median AND dividing by 1,000 lands it
# inside the range of normal training values, widened by si_margin either side
si_ratio, si_margin = 100, 2

# Columns never used as model features
drop_cols = ["exposure_kaggle"]   # supplied exposure: contains planted errors
# ---------------------------------------------------------------------------

# Documented categories, from the dataset description (after normalising text)
band_labels = [b[2] for b in age_bands]
regions = {"developed": [f"r{i:02d}" for i in range(1, 13)],
           "emerging":  [f"e{i:02d}" for i in range(1, 7)]}
categories_common = {
    "marital_status": ["single", "married", "divorced", "widowed"],
    "credit_band": ["a", "b", "c", "d", "e"],
    "gender": ["m", "f"],
    "urban_density": ["rural", "suburban", "urban", "dense_urban"],
    "soil_type": ["clay", "sand", "loam", "chalk", "rock"],
    "property_type": ["detached", "semi_detached", "terraced", "flat"],
    "roof_type": ["pitched", "flat"],
    "energy_rating": ["a", "b", "c", "d", "e", "f", "g"],
    "channel": ["direct", "comparison_site", "broker"],
    "payment_method": ["annual", "monthly"],
    "body_type": ["hatchback", "saloon", "estate", "suv", "coupe"],
    "transmission": ["manual", "automatic"],
    "fuel_type": ["petrol", "diesel", "hybrid", "electric"],
    "cover_type": ["comprehensive", "third_party_fire_theft", "third_party_only"],
    "overnight_parking": ["garage", "driveway", "street"],
    "vehicle_use": ["social", "commuting", "business"],
    "construction_type": ["brick", "stone", "timber", "non_standard"],
    "occupancy": ["owner_occupied", "let"],
    "tenure": ["owner", "renter"],
    "security_level": ["standard", "locks", "alarm_and_locks"],
    "driver_age_band": band_labels,
    "policyholder_age_band": band_labels,
}

def categories_for(market):
    """Documented categories for one market (only the region codes differ)."""
    return {**categories_common, "region": regions[market]}

import os, gc, numpy as np, pandas as pd

work_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
P2 = {}   # everything Phase 2 builds, per market
for m in markets:
    d_in, d_out = os.path.join(work_dir, in_prefix + m), os.path.join(work_dir, out_prefix + m)
    if not os.path.isdir(d_in):
        raise RuntimeError(f"No Phase 1 output in {d_in} - run Phase 1 first")
    os.makedirs(d_out, exist_ok=True)
    P2[m] = {"in_dir": d_in, "out_dir": d_out}
    print(f"{m:<10} Phase 1 input: {d_in} | Phase 2 output: {d_out}")

developed  Phase 1 input: /kaggle/working/prep_developed | Phase 2 output: /kaggle/working/clean_developed
emerging   Phase 1 input: /kaggle/working/prep_emerging | Phase 2 output: /kaggle/working/clean_emerging


In [12]:
# %% Phase 2 Cell 2: Cleaning function ========================================

def norm(s):
    """Lowercase, trim, and turn spaces/hyphens into '_' (keeps blanks blank)."""
    return s.astype("string").str.strip().str.lower().str.replace(r"[ -]+", "_", regex=True)

def is_text(s):
    return (pd.api.types.is_string_dtype(s) or s.dtype == object) and not pd.api.types.is_bool_dtype(s) \
        and s.dropna().map(lambda v: isinstance(v, str)).all()

def to_band(age):
    out = pd.Series(pd.NA, index=age.index, dtype="string")
    for lo, hi, lab in age_bands:
        out[(age >= lo) & (age <= hi)] = lab
    return out

def clean_line(tr, te, line, market):
    categories = categories_for(market)
    log, fills = [], []
    d = pd.concat([tr.assign(_set="train"), te.assign(_set="test")], ignore_index=True)
    d = d.drop(columns=[c for c in drop_cols if c in d.columns])
    is_tr = d["_set"] == "train"

    def note(column, action, idx):
        idx = pd.Series(idx, index=d.index).fillna(False).astype(bool)
        n_tr, n_te = int((idx & is_tr).sum()), int((idx & ~is_tr).sum())
        if n_tr + n_te:
            log.append(dict(market=market, line=line, column=column, action=action, train_rows=n_tr, test_rows=n_te))

    text_cols = [c for c in d.columns if c != "_set" and not c.endswith("_id") and is_text(d[c])]

    # 1. Standardise text
    for c in text_cols:
        d[c] = norm(d[c])

    # 2. Map known undocumented / local codes
    for c, m in code_map.items():
        if c in d.columns:
            idx = d[c].isin(list(m))
            note(c, "mapped " + ", ".join(f"{k}->{v}" for k, v in m.items() if (d[c] == k).any()), idx)
            d.loc[idx, c] = d.loc[idx, c].map(m)

    # 3. Free-text occupation: split the "self-employed" prefix into a flag
    if "occupation" in d.columns:
        se = d["occupation"].str.startswith("self_employed_").fillna(False)
        note("occupation", "\"self-employed\" prefix split into self_employed flag", se)
        d["self_employed"] = np.where(d["occupation"].isna(), pd.NA, se.map({True: "true", False: "false"}))
        d.loc[se, "occupation"] = d.loc[se, "occupation"].str.replace("^self_employed_", "", regex=True)
        text_cols.append("self_employed")

    # 4. Same age bands in both markets (developed has exact ages)
    for age_col, band_col in [("driver_age", "driver_age_band"), ("policyholder_age", "policyholder_age_band")]:
        if age_col in d.columns and band_col not in d.columns:
            d[band_col] = to_band(d[age_col])
            note(band_col, f"derived from {age_col}", d[band_col].notna())
            text_cols.append(band_col)

    # 5. Stop on any undocumented value not covered by code_map
    for c, allowed in categories.items():
        if c in d.columns:
            bad = d[c].notna() & ~d[c].isin(allowed)
            if bad.any():
                print(pd.crosstab(d.loc[bad, c], d.loc[bad, "_set"]))
                raise RuntimeError(f"{market} {line}: {c} has undocumented values: {sorted(d.loc[bad, c].unique())}. "
                                   "Check them, then add each to code_map in Phase 2 Cell 1.")

    # 6. sum_insured entered 1,000x too large (unit error), in train and test alike
    if "sum_insured" in d.columns:
        si_tr = d.loc[is_tr, "sum_insured"]
        med = si_tr.median()
        normal = si_tr[si_tr <= si_ratio * med]
        lo, hi = normal.min() / si_margin, normal.max() * si_margin
        idx = (d["sum_insured"] > si_ratio * med) & (d["sum_insured"] / 1000).between(lo, hi)
        note("sum_insured", "divided by 1,000 (entered in units, not thousands) + flag", idx)
        d["sum_insured_fixed"] = idx
        d.loc[idx, "sum_insured"] = d.loc[idx, "sum_insured"] / 1000
        left = d["sum_insured"] > si_ratio * med
        if left.any():
            raise RuntimeError(f"{line}: {int(left.sum())} sum_insured values still look wrong after the fix - inspect them")

    # 7. Blanks with a stated meaning, then other text blanks -> unknown
    for c, v in na_meaning.items():
        if c in d.columns:
            idx = d[c].isna(); note(c, f'blank -> "{v}"', idx); d.loc[idx, c] = v
    for c in text_cols:
        idx = d[c].isna()
        note(c, f'blank -> "{unknown_label}"', idx)
        d[c] = d[c].fillna(unknown_label).astype(str)

    # 8. Yes/no columns: with blanks -> "true" / "false" / "unknown"
    for c in [c for c in d.columns if c != "_set" and c not in text_cols]:
        s = d[c]
        vals = set(s.dropna().unique())
        if (pd.api.types.is_bool_dtype(s) or (s.dtype == object and vals <= {True, False})) and s.isna().any():
            idx = s.isna(); note(c, f'yes/no blank -> "{unknown_label}"', idx)
            d[c] = s.map({True: "true", False: "false"}).fillna(unknown_label).astype(str)
        elif s.dtype == object and vals <= {True, False}:
            d[c] = s.astype(bool)

    # 9. Consistency caps (formulas, not fixed values)
    if "years_licensed" in d.columns:
        if "driver_age" in d.columns:
            oldest, how = d["driver_age"], "driver_age"
        else:   # banded market: the oldest age in the driver's band
            oldest, how = d["driver_age_band"].map({lab: hi for lo, hi, lab in age_bands}), "top of driver_age_band"
        max_lic = (oldest - min_licence_age).clip(lower=0)
        idx = d["years_licensed"].notna() & max_lic.notna() & (d["years_licensed"] > max_lic)
        note("years_licensed", f"capped at {how} - {min_licence_age}", idx)
        d["years_licensed_capped"] = idx
        d.loc[idx, "years_licensed"] = max_lic[idx]
    if {"ncb_years", "years_licensed"} <= set(d.columns):
        idx = d["ncb_years"].notna() & d["years_licensed"].notna() & (d["ncb_years"] > d["years_licensed"])
        note("ncb_years", "capped at years_licensed", idx)
        d["ncb_capped"] = idx
        d.loc[idx, "ncb_years"] = d.loc[idx, "years_licensed"]

    # 10. Telematics score: blank by design when not opted in (developed only)
    if "telematics_score" in d.columns:
        d["telematics_score_known"] = d["telematics_score"].notna()
        fill = d.loc[is_tr, "telematics_score"].median()
        idx = d["telematics_score"].isna()
        note("telematics_score", f"blank -> opted-in training median {fill:.1f}", idx)
        fills.append(dict(market=market, line=line, column="telematics_score", fill_value=fill))
        d.loc[idx, "telematics_score"] = fill

    # 11. Numeric blanks -> training median + flag (repeated after derived features)
    def impute(d):
        for c in [c for c in d.columns if pd.api.types.is_numeric_dtype(d[c]) and not pd.api.types.is_bool_dtype(d[c])]:
            if d[c].isna().any():
                fill = d.loc[is_tr, c].median()
                if pd.isna(fill):
                    raise RuntimeError(f"{line}: {c} has no training values to fill from")
                idx = d[c].isna()
                note(c, f"blank -> training median {fill:g} + flag", idx)
                fills.append(dict(market=market, line=line, column=c, fill_value=fill))
                d[c + "_missing"] = idx
                d[c] = d[c].fillna(fill)
        return d
    d = impute(d)

    # 12. Derived features from each row's own dates
    d["policy_year"] = d["period_start"].dt.year
    if "date_of_birth" in d.columns:
        dob, ps = d["date_of_birth"], d["period_start"]
        before_bday = (ps.dt.month < dob.dt.month) | ((ps.dt.month == dob.dt.month) & (ps.dt.day < dob.dt.day))
        d["customer_age"] = ps.dt.year - dob.dt.year - before_bday.astype(int)
    if "customer_since_year" in d.columns: d["customer_tenure"] = d["policy_year"] - d["customer_since_year"]
    if "year_built" in d.columns:          d["property_age"] = d["policy_year"] - d["year_built"]
    d = impute(d)

    tr_out = d[is_tr].drop(columns="_set").reset_index(drop=True)
    te_out = d[~is_tr].drop(columns="_set").reset_index(drop=True)
    return tr_out, te_out, pd.DataFrame(log), pd.DataFrame(fills)

In [13]:
# %% Phase 2 Cell 3: Load Phase 1 output and clean each line, for each market ==

load_in = lambda m, name: pd.read_parquet(os.path.join(P2[m]["in_dir"], name + ".parquet"))
pd.set_option("display.width", 250, "display.max_colwidth", 80, "display.max_rows", 500)

for m in markets:
    logs, fill_list = [], []
    P2[m]["train"], P2[m]["test"] = {}, {}
    for l in lines:
        print(f"Cleaning {m} {l}")
        tr, te, lg, fv = clean_line(load_in(m, f"train_{l}"), load_in(m, f"test_{l}"), l, m)
        P2[m]["train"][l], P2[m]["test"][l] = tr, te
        logs.append(lg); fill_list.append(fv)
    P2[m]["log"]   = pd.concat(logs, ignore_index=True)
    P2[m]["fills"] = pd.concat(fill_list, ignore_index=True)

cleaning_log = pd.concat([P2[m]["log"] for m in markets], ignore_index=True)
print("\nCleaning log, both markets (rows changed by each rule)")
print(cleaning_log.to_string(index=False))

Cleaning developed motor
Cleaning developed buildings
Cleaning developed contents
Cleaning emerging motor
Cleaning emerging buildings
Cleaning emerging contents

Cleaning log, both markets (rows changed by each rule)
   market      line                column                                                        action  train_rows  test_rows
developed     motor       driver_age_band                                       derived from driver_age      979968     254986
developed     motor     overnight_parking                                            blank -> "unknown"       24371       6398
developed     motor            occupation                                            blank -> "unknown"       24158       6307
developed     motor        marital_status                                            blank -> "unknown"       24732       6442
developed     motor             soil_type                                            blank -> "unknown"       24741       6524
developed     motor  

In [14]:
# %% Phase 2 Cell 4: Final checks and save, for each market ===================

date_like = lambda d: [c for c in d.columns if pd.api.types.is_datetime64_any_dtype(d[c])]

for m in markets:
    for l in lines:
        tr, te = P2[m]["train"][l], P2[m]["test"][l]
        # No blanks left outside date columns
        for name, d in [("train", tr), ("test", te)]:
            na_left = [c for c in d.columns if c not in date_like(d) and d[c].isna().any()]
            if na_left:
                raise RuntimeError(f"{m} {l} {name}: blanks remain in {na_left}")
        # Only documented values (plus unknown) in category columns
        for c, allowed in categories_for(m).items():
            if c in tr.columns:
                bad = set(pd.concat([tr[c], te[c]]).unique()) - set(allowed) - {unknown_label}
                if bad:
                    raise RuntimeError(f"{m} {l}: {c} still has {sorted(bad)}")
        # Train and test: same columns, same types
        if list(tr.columns) != list(te.columns):
            raise RuntimeError(f"{m} {l}: train and test columns differ")
        diff = [c for c in tr.columns if tr[c].dtype != te[c].dtype]
        if diff:
            raise RuntimeError(f"{m} {l}: column types differ: {diff}")
        # Row counts unchanged
        assert len(tr) == len(load_in(m, f"train_{l}")) and len(te) == len(load_in(m, f"test_{l}"))

        print(f"  OK: {m:<10} {l:<10} train {len(tr):,} rows, test {len(te):,} rows, {tr.shape[1]} columns, "
              "no blanks, only documented categories")
        tr.to_parquet(os.path.join(P2[m]["out_dir"], f"train_{l}.parquet"), index=False)
        te.to_parquet(os.path.join(P2[m]["out_dir"], f"test_{l}.parquet"), index=False)
    P2[m]["log"].to_csv(os.path.join(P2[m]["out_dir"], "cleaning_log.csv"), index=False)
    P2[m]["fills"].to_csv(os.path.join(P2[m]["out_dir"], "fill_values.csv"), index=False)
for m in markets:
    print(f"\nPhase 2 complete for {m}. Files in {P2[m]['out_dir']}:", sorted(os.listdir(P2[m]["out_dir"])))

  OK: developed  motor      train 979,968 rows, test 254,986 rows, 90 columns, no blanks, only documented categories
  OK: developed  buildings  train 707,726 rows, test 184,155 rows, 74 columns, no blanks, only documented categories
  OK: developed  contents   train 721,617 rows, test 187,498 rows, 75 columns, no blanks, only documented categories
  OK: emerging   motor      train 47,283 rows, test 90,521 rows, 82 columns, no blanks, only documented categories
  OK: emerging   buildings  train 27,447 rows, test 52,295 rows, 71 columns, no blanks, only documented categories
  OK: emerging   contents   train 34,566 rows, test 66,247 rows, 72 columns, no blanks, only documented categories

Phase 2 complete for developed. Files in /kaggle/working/clean_developed: ['cleaning_log.csv', 'fill_values.csv', 'test_buildings.parquet', 'test_contents.parquet', 'test_motor.parquet', 'train_buildings.parquet', 'train_contents.parquet', 'train_motor.parquet']

Phase 2 complete for emerging. Files in

In [15]:
# %% Phase 2 Cell 5: Are the blanks random? (read-only check) =================
# Blanks are filled with "unknown" or the median plus a flag. That is only safe
# if blank rows claim like everyone else, or if the flag captures the difference.
# For each column with blanks, compare training claim frequency for blank vs
# filled rows. A large gap on many claims means the blank itself carries
# information (keep the flag as a model feature and mention it in the log).

rows = []
for m in markets:
    cm = pd.read_parquet(os.path.join(P2[m]["in_dir"], "claims_matched.parquet"))
    cm = cm[cm["status"] != "repudiated"]
    for l in lines:
        tr = P2[m]["train"][l]
        n = tr["row_id"].map(cm[cm["line"] == l].groupby("row_id").size()).fillna(0)
        expo = tr["exposure"]
        flags = [c for c in tr.columns if c.endswith("_missing")]
        unknowns = [c for c in tr.columns if (tr[c].dtype == object or pd.api.types.is_string_dtype(tr[c]))
                    and (tr[c].astype(str) == unknown_label).any()]
        for c in flags + unknowns:
            blank = tr[c].astype(bool) if c in flags else tr[c].astype(str) == unknown_label
            if blank.sum() == 0 or (~blank).sum() == 0:
                continue
            f_b, f_f = n[blank].sum() / expo[blank].sum(), n[~blank].sum() / expo[~blank].sum()
            se = np.sqrt(n[blank].sum()) / expo[blank].sum()   # Poisson standard error, blank group
            rows.append(dict(market=m, line=l, column=c.removesuffix("_missing"), blank_share=round(blank.mean(), 3),
                             claims_in_blank=int(n[blank].sum()), freq_blank=round(f_b, 4), freq_filled=round(f_f, 4),
                             ratio=round(f_b / f_f, 3), z=round((f_b - f_f) / se, 1) if se > 0 else np.nan))
blank_check = pd.DataFrame(rows)
blank_check["looks_informative"] = blank_check["z"].abs() > 3
print(blank_check.to_string(index=False))
for m in markets:
    blank_check[blank_check["market"] == m].to_csv(os.path.join(P2[m]["out_dir"], "blank_check.csv"), index=False)
print("\nColumns where blanks look informative (|z| > 3):",
      blank_check.loc[blank_check["looks_informative"], ["market", "line", "column"]].values.tolist() or "none")

   market      line               column  blank_share  claims_in_blank  freq_blank  freq_filled  ratio    z  looks_informative
developed     motor          vehicle_age        0.025             2772      0.1408       0.1409  1.000 -0.0              False
developed     motor       engine_size_cc        0.025             2715      0.1367       0.1410  0.970 -1.6              False
developed     motor       annual_mileage        0.025             2713      0.1367       0.1410  0.970 -1.6              False
developed     motor       household_size        0.025             2625      0.1350       0.1410  0.957 -2.3              False
developed     motor     years_at_address        0.025             2759      0.1388       0.1409  0.985 -0.8              False
developed     motor           flood_zone        0.029             3299      0.1418       0.1409  1.007  0.4              False
developed     motor distance_to_river_km        0.025             2731      0.1393       0.1409  0.989 -0.6    

In [16]:
# %% Phase 2 Cell 6: 2023 system change - did other columns shift? (read-only)
# The host confirmed a 2023 system change "affecting some codes" (t -> w). This
# checks every column of the Phase 1 output for a break between consecutive
# years: category values whose share jumps or that appear/vanish, and numeric
# medians that move sharply. The developed market (2021-24) spans the change.
# Some flags are genuine (weather, renewal_count): look for breaks AT the
# change year with no business explanation.

max_levels, share_jump, ratio_jump = 30, 0.03, 1.25
skip = {"row_id", "orig_row_id", "policy_id", "customer_id", "address_id",
        "period_start", "period_end", "exposure", "exposure_kaggle", "weather_year"}
for m, l in [(m, l) for m in markets for l in lines]:
    d = load_in(m, f"train_{l}")
    yr = d["period_start"].dt.year
    if yr.nunique() < 2:
        print(m, l, ": only one year, nothing to compare"); continue
    print(f"\n===== {m} {l} =====")
    for c in [c for c in d.columns if c not in skip]:
        x = d[c]
        n_val = x.nunique(dropna=False)
        if pd.api.types.is_numeric_dtype(x) and not pd.api.types.is_bool_dtype(x) and n_val > max_levels:
            med = x.groupby(yr).median()
            r = (med / med.shift(1)).dropna()
            if ((r > ratio_jump) | (r < 1 / ratio_jump)).any():
                print(f"  {c:<26} median by year: " + "  ".join(f"{y}={v:.4g}" for y, v in med.items()))
        elif n_val <= max_levels:
            tab = pd.crosstab(yr, x.astype("string").fillna("<blank>"), normalize="index")
            jump = tab.diff().abs().max() > share_jump
            on_off = (tab.min() == 0) & (tab.max() > 0.005)
            hit = tab.columns[jump | on_off]
            if len(hit):
                print(f"\n  {c} - share of each value by year (%):")
                print((100 * tab[hit]).round(1).to_string())


===== developed motor =====

  ncb_years - share of each value by year (%):
ncb_years        0    15
period_start            
2021          10.0   5.4
2022           5.9   9.3
2023           5.2  12.0
2024           4.8  13.9

  vehicle_age - share of each value by year (%):
vehicle_age   1.0  2.0
period_start          
2021          5.4  8.8
2022          1.3  5.8
2023          0.8  2.4
2024          0.8  2.0

  new_business - share of each value by year (%):
new_business  False  True
period_start             
2021           88.0  12.0
2022           84.8  15.2
2023           84.7  15.3
2024           84.6  15.4

  renewal_count - share of each value by year (%):
renewal_count     0     1   16   17   18     2
period_start                                  
2021           12.0   5.8  0.0  0.0  0.0   5.8
2022           15.2  10.2  5.0  0.0  0.0   5.0
2023           15.3  15.2  4.1  4.1  0.0   8.3
2024           15.4  15.3  3.3  3.4  3.4  12.4

  customer_since_year - share of each value

In [17]:
# %% Phase 2 Cell 7: Reconcile with the Week 1 R cleaning, then free memory ====
week1_r = {("buildings", "construction_type"): 85556,
           ("motor", "years_licensed"): 2962,
           ("motor", "ncb_years"): 127106}

if "developed" in P2:
    lg = P2["developed"]["log"]
    py = {k: int(lg.loc[(lg["line"] == k[0]) & (lg["column"] == k[1]), "train_rows"].sum()) for k in week1_r}
    recon = pd.DataFrame({"week1_R": week1_r, "python": py})
    recon["match"] = recon["week1_R"] == recon["python"]
    print(recon.to_string())
    if not recon["match"].all():
        raise RuntimeError("Python cleaning does not reproduce the Week 1 R counts - investigate")
    print("OK: Python cleaning reproduces the Week 1 R counts")
else:
    print("Developed market not run - no reconciliation with Week 1.")

# Phase 3 reads the saved files, so the large in-memory tables can go
P2 = {m: {"in_dir": P2[m]["in_dir"], "out_dir": P2[m]["out_dir"]} for m in P2}
gc.collect()
print("Phase 2 tables cleared from memory (all outputs are saved to disk).")

                             week1_R  python  match
buildings construction_type    85556   85556   True
motor     years_licensed        2962    2962   True
          ncb_years           127106  127106   True
OK: Python cleaning reproduces the Week 1 R counts
Phase 2 tables cleared from memory (all outputs are saved to disk).


Phase 3

In [18]:
# =============================================================================
# PART 2 - PHASE 3: BUILDING THE TARGET  (Python port of the Part 1 R Phase 3,
# plus borrowing from the developed market)
#
# Target (from the brief): final cost of all claims with a loss date in the
# row's period, net of the policy excess and net of recoveries, excluding
# declined claims. Steps:
#   1. Cost of each claim (incurred)   2. Excess check (amounts already net)
#   3. Recoveries   4. Declined claims   5. Sum per row
#   6. Develop to ultimate   7. Trend   8. Large claims   9. Checks
#
# Runs BOTH markets in one go: each cell loops over `markets`, developed first
# (the emerging run borrows from it within the same cell):
#   - developed: same method as Week 1; Cell 9 reconciles with the Week 1 R run
#   - emerging:  two years and ~1,000 property claims are too few to measure
#     late development, trend and the large-claim loading alone, so Steps 6-8
#     blend the emerging estimate with the developed one by credibility, or
#     borrow the developed estimate where the emerging one can't be measured.
#
# Reproducible: no years, dates or amounts are typed in. The valuation date is
# read from the claims file; everything else is calculated from the data.
# =============================================================================

# %% Phase 3 Cell 1: Settings and helpers ======================================

# ---- SETTINGS: the only lines to change for a different dataset -----------
markets        = ["developed", "emerging"]   # developed FIRST: emerging borrows its estimates
source_market  = "developed"                 # the market the others borrow from
lines          = ["motor", "buildings", "contents"]
# ---------------------------------------------------------------------------

import os, gc, warnings
import numpy as np
import pandas as pd

if markets[0] != source_market:
    raise RuntimeError(f"'{source_market}' must be first in markets - the other markets borrow from it")

work_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
read_out = lambda d, name: pd.read_parquet(os.path.join(d, name + ".parquet"))
fmt = lambda x, d=0: f"{x:,.{d}f}"

def show(df, money=(), pct=(), dec=None):
    """Print a table with amounts as 1,234 and rates as +1.23% (formatting done before printing)."""
    out = df.copy()
    for c in money:
        if c in out: out[c] = out[c].map(lambda x: fmt(x) if pd.notna(x) else "")
    for c in pct:
        if c in out: out[c] = out[c].map(lambda x: f"{100 * x:+.2f}%" if pd.notna(x) else "")
    if dec: out = out.round(dec)
    print(out.to_string(index=False))
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_rows", 500)

borrow_dir = os.path.join(work_dir, f"target_{source_market}")
P3 = {}   # per market: folders, and results passed between cells
for m in markets:
    P3[m] = {"prep_dir": os.path.join(work_dir, f"prep_{m}"), "clean_dir": os.path.join(work_dir, f"clean_{m}"),
             "target_dir": os.path.join(work_dir, f"target_{m}"), "borrowing": m != source_market}
    for k in ("prep_dir", "clean_dir"):
        if not os.path.isdir(P3[m][k]):
            raise RuntimeError(f"Missing {P3[m][k]} - run Phases 1 and 2 first")
    os.makedirs(P3[m]["target_dir"], exist_ok=True)
    print(f"{m:<10} inputs: {P3[m]['prep_dir']}, {P3[m]['clean_dir']} | output: {P3[m]['target_dir']}"
          f" | borrows from: {borrow_dir if P3[m]['borrowing'] else 'nothing (source market)'}")

developed  inputs: /kaggle/working/prep_developed, /kaggle/working/clean_developed | output: /kaggle/working/target_developed | borrows from: nothing (source market)
emerging   inputs: /kaggle/working/prep_emerging, /kaggle/working/clean_emerging | output: /kaggle/working/target_emerging | borrows from: /kaggle/working/target_developed


In [19]:
# %% Phase 3 Cell 2 (Step 1): Cost of each claim, for each market ============
# incurred_to_date = paid so far + current estimate for open claims.
# Only claims matched to a training row in Phase 1 are used.

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    claims_m = read_out(prep_dir3, "claims_matched")

    # Valuation date: the latest report date in the claims file (not typed in)
    valuation_date = claims_m["report_date"].max()
    print("Valuation date (latest report date in the data):", valuation_date.date(), "\n")

    # Amounts must be present and sensible before anything is built on them
    amount_cols = ["paid_to_date", "incurred_to_date", "recoveries"]
    na_amounts = claims_m[amount_cols].isna().sum()
    if na_amounts.any():
        raise RuntimeError(f"Blank claim amounts: {na_amounts[na_amounts > 0].to_dict()}")
    if (claims_m["incurred_to_date"] < 0).any():                                raise RuntimeError("Negative incurred_to_date found")
    if (claims_m["incurred_to_date"] < claims_m["paid_to_date"] - 0.01).any():  raise RuntimeError("incurred_to_date below paid_to_date found")
    if (claims_m["loss_date"] > valuation_date).any():                          raise RuntimeError("Loss dates after the valuation date found")

    claim_costs = pd.DataFrame({
        "claim_id": claims_m["claim_id"], "row_id": claims_m["row_id"], "orig_row_id": claims_m["orig_row_id"],
        "line": claims_m["line"], "peril": claims_m["peril"], "status": claims_m["status"],
        "loss_date": claims_m["loss_date"], "report_date": claims_m["report_date"],
        "loss_year": claims_m["loss_date"].dt.year,
        "report_delay_d": (claims_m["report_date"] - claims_m["loss_date"]).dt.days,
        "paid_to_date": claims_m["paid_to_date"], "incurred_to_date": claims_m["incurred_to_date"],
        "recoveries": claims_m["recoveries"],
        "open_estimate": claims_m["incurred_to_date"] - claims_m["paid_to_date"],   # still to be paid on open claims
        "cost_incurred": claims_m["incurred_to_date"],                              # Step 1 result
    })

    g = claim_costs.groupby(["line", "loss_year"])
    step1_summary = pd.DataFrame({
        "claims": g.size(), "closed": g["status"].apply(lambda s: (s == "closed").sum()),
        "open": g["status"].apply(lambda s: (s == "open").sum()),
        "declined": g["status"].apply(lambda s: (s == "repudiated").sum()),
        "paid": g["paid_to_date"].sum(), "open_estimate": g["open_estimate"].sum(), "incurred": g["cost_incurred"].sum(),
    }).reset_index()
    step1_summary["avg_incurred"] = step1_summary["incurred"] / step1_summary["claims"]
    step1_summary["open_share_of_incurred"] = (step1_summary["open_estimate"] / step1_summary["incurred"]).round(4)

    print("Step 1: claim cost (incurred) by line and loss year")
    show(step1_summary, money=["paid", "open_estimate", "incurred", "avg_incurred"])
    print(f"\nClaims: {len(claim_costs):,} | total incurred: {fmt(claim_costs['cost_incurred'].sum())}")

    # Claims-table scan for the 2023 system change: peril and status mix by loss year
    print("\nClaims-table scan: share of claims by peril and loss year (%) - look for a break at one year")
    print((100 * pd.crosstab([claim_costs["line"], claim_costs["peril"]], claim_costs["loss_year"],
                             normalize="columns")).round(1).to_string())
    print("\nShare of claims by status and loss year (%)")
    print((100 * pd.crosstab(claim_costs["status"], claim_costs["loss_year"], normalize="columns")).round(1).to_string())
    P3[market]["claim_costs"] = claim_costs
    P3[market]["valuation_date"] = valuation_date


============================== DEVELOPED ==============================
Valuation date (latest report date in the data): 2024-12-30 

Step 1: claim cost (incurred) by line and loss year
     line  loss_year  claims  closed  open  declined        paid open_estimate    incurred avg_incurred  open_share_of_incurred
buildings       2021    6195    6178    17         0  53,836,682       216,152  54,052,834        8,725                  0.0040
buildings       2022    6792    6737    55         0  60,477,776     1,051,555  61,529,331        9,059                  0.0171
buildings       2023    6817    6619   198         0  61,202,630     3,629,842  64,832,471        9,510                  0.0560
buildings       2024    6571    4951  1620         0  48,423,038    18,930,911  67,353,950       10,250                  0.2811
 contents       2021    5645    5583     0        62  14,870,108             0  14,870,108        2,634                  0.0000
 contents       2022    6097    6024     0   

In [20]:
# %% Phase 3 Cell 3 (Step 2): Excess - check, don't subtract ====================
# The description says paid_to_date is already net of excess, and incurred is
# paid + estimate. Subtracting the excess again would count it twice, so this
# step only checks the data behaves as "net of excess" would: a higher excess
# means fewer claims and less paid per policy-year (cost PER CLAIM can still
# rise, because the small claims drop out).

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    claim_costs = P3[market]["claim_costs"]
    row_excess = pd.concat([read_out(clean_dir3, f"train_{l}")[["row_id", "exposure", "voluntary_excess", "compulsory_excess"]]
                            .assign(line=l) for l in lines], ignore_index=True)

    claim_costs = claim_costs.drop(columns=[c for c in ["voluntary_excess", "compulsory_excess", "total_excess"] if c in claim_costs])
    claim_costs = claim_costs.merge(row_excess.drop(columns=["exposure", "line"]), on="row_id", how="left")
    claim_costs["total_excess"] = claim_costs["voluntary_excess"] + claim_costs["compulsory_excess"]
    if claim_costs["total_excess"].isna().any():
        raise RuntimeError("Some claims have no excess from their policy row")

    # Check A: per policy-year, by voluntary excess (paying claims only)
    paying_a = claim_costs[claim_costs["status"] != "repudiated"].groupby("row_id").agg(n_claims=("claim_id", "size"), cost=("cost_incurred", "sum"))
    ea = row_excess.merge(paying_a, on="row_id", how="left").fillna({"n_claims": 0, "cost": 0})
    excess_check = ea.groupby(["line", "voluntary_excess"]).agg(policy_years=("exposure", "sum"), n=("n_claims", "sum"), cost=("cost", "sum")).reset_index()
    excess_check["claim_freq"] = (excess_check["n"] / excess_check["policy_years"]).round(4)
    excess_check["cost_per_year"] = excess_check["cost"] / excess_check["policy_years"]
    excess_check["cost_per_claim"] = excess_check["cost"] / excess_check["n"].clip(lower=1)
    smallest = claim_costs[claim_costs["status"] == "closed"].groupby(["line", "voluntary_excess"])["cost_incurred"].quantile(0.01).rename("smallest_1pct")
    excess_check = excess_check.merge(smallest.reset_index(), on=["line", "voluntary_excess"], how="left").drop(columns=["n", "cost"])

    # Check B: closed, paying claims smaller than their excess (impossible if amounts were gross)
    closed = claim_costs[claim_costs["status"] == "closed"]
    below_excess = closed.assign(
        zero_cost=closed["cost_incurred"] == 0,
        positive_below_excess=(closed["cost_incurred"] > 0) & (closed["cost_incurred"] < closed["total_excess"])
    ).groupby("line").agg(closed_claims=("claim_id", "size"), zero_cost=("zero_cost", "sum"),
                          positive_below_excess=("positive_below_excess", "sum")).reset_index()
    below_excess["share_positive_below"] = (below_excess["positive_below_excess"] / below_excess["closed_claims"]).round(4)

    print("Step 2, check A: by voluntary excess. If amounts are net, claim_freq and cost_per_year fall as excess rises.")
    show(excess_check, money=["policy_years", "cost_per_year", "cost_per_claim", "smallest_1pct"])
    print("\nStep 2, check B: closed claims with cost above zero but below the row's total excess")
    show(below_excess)
    print("\nDecision: amounts are used as supplied (already net of excess); no excess is subtracted.")

    claim_costs.to_parquet(os.path.join(target_dir, "claim_costs_step2.parquet"), index=False)
    step1_summary.to_csv(os.path.join(target_dir, "step1_summary.csv"), index=False)
    excess_check.to_csv(os.path.join(target_dir, "step2_excess_check.csv"), index=False)
    P3[market]["claim_costs"] = claim_costs


============================== DEVELOPED ==============================
Step 2, check A: by voluntary excess. If amounts are net, claim_freq and cost_per_year fall as excess rises.
     line  voluntary_excess policy_years  claim_freq cost_per_year cost_per_claim smallest_1pct
buildings                 0      295,675      0.0481           432          8,975            32
buildings               250      176,999      0.0447           423          9,466            37
buildings               500       88,705      0.0369           397         10,748            34
buildings              1000       29,180      0.0330           343         10,389            41
 contents                 0      295,160      0.0460           119          2,581            13
 contents               250      177,055      0.0388           116          2,981            14
 contents               500       89,091      0.0292            85          2,925            14
 contents              1000       29,554      0.02

In [21]:
# %% Phase 3 Cell 4 (Step 3): Take off recoveries ==============================
# cost_net = incurred_to_date - recoveries. Recoveries arrive late, so recent
# loss years show less recovered so far; Step 6 develops the NET cost, so this
# step only takes off what has been recovered to date.

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    claim_costs = P3[market]["claim_costs"]
    claim_costs["cost_net"] = claim_costs["cost_incurred"] - claim_costs["recoveries"]   # Step 3 result
    if (claim_costs["recoveries"] < 0).any():
        raise RuntimeError("Negative recoveries found")
    n_neg = int((claim_costs["cost_net"] < -0.01).sum())
    if n_neg:
        raise RuntimeError(f"{n_neg} claims would have negative net cost (recoveries above incurred) - inspect before continuing")
    assert abs(claim_costs["cost_incurred"].sum() - claim_costs["recoveries"].sum() - claim_costs["cost_net"].sum()) < 1

    def recov_table(keys):
        g = claim_costs.groupby(keys)
        t = pd.DataFrame({"claims": g.size(), "with_recovery": g["recoveries"].apply(lambda s: (s > 0).sum()),
                          "incurred": g["cost_incurred"].sum(), "recoveries": g["recoveries"].sum(),
                          "net": g["cost_net"].sum()}).reset_index()
        t["recovery_rate"] = (t["recoveries"] / t["incurred"]).round(4)
        return t
    step3_by_year, step3_by_peril = recov_table(["line", "loss_year"]), recov_table(["line", "peril"])

    print("Step 3: recoveries by line and loss year"); show(step3_by_year, money=["incurred", "recoveries", "net"])
    print("\nStep 3: recoveries by peril"); show(step3_by_peril.sort_values(["line", "recovery_rate"], ascending=[True, False]), money=["incurred", "recoveries", "net"])
    tot = claim_costs[["cost_incurred", "recoveries", "cost_net"]].sum()
    print(f"\nTotal incurred {fmt(tot.cost_incurred)} - recoveries {fmt(tot.recoveries)} = net {fmt(tot.cost_net)} "
          f"({100 * tot.recoveries / tot.cost_incurred:.2f}% recovered)")

    claim_costs.to_parquet(os.path.join(target_dir, "claim_costs_step3.parquet"), index=False)
    step3_by_year.to_csv(os.path.join(target_dir, "step3_recoveries_by_year.csv"), index=False)
    step3_by_peril.to_csv(os.path.join(target_dir, "step3_recoveries_by_peril.csv"), index=False)
    P3[market]["claim_costs"] = claim_costs


============================== DEVELOPED ==============================
Step 3: recoveries by line and loss year
     line  loss_year  claims  with_recovery    incurred recoveries         net  recovery_rate
buildings       2021    6195              0  54,052,834          0  54,052,834         0.0000
buildings       2022    6792              0  61,529,331          0  61,529,331         0.0000
buildings       2023    6817              0  64,832,471          0  64,832,471         0.0000
buildings       2024    6571              0  67,353,950          0  67,353,950         0.0000
 contents       2021    5645              0  14,870,108          0  14,870,108         0.0000
 contents       2022    6097              0  16,040,969          0  16,040,969         0.0000
 contents       2023    6235              0  17,539,467          0  17,539,467         0.0000
 contents       2024    6075              0  17,516,035          0  17,516,035         0.0000
    motor       2021   27711           3

In [22]:
# %% Phase 3 Cell 5 (Steps 4-5): Declined claims, then cost per policy row =====
# Declined (repudiated) claims are excluded by the brief: cost zero, not counted.
# Each row's paying claims are added up; rows with no claims get zero (a real
# and very common answer). Totals are also kept by peril for Step 6.

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    claim_costs = P3[market]["claim_costs"]
    claim_costs["declined"] = claim_costs["status"] == "repudiated"
    claim_costs["cost_target"] = np.where(claim_costs["declined"], 0.0, claim_costs["cost_net"])   # Step 4 result

    dec = claim_costs.assign(declined_incurred=claim_costs["cost_incurred"].where(claim_costs["declined"], 0),
                             declined_net=claim_costs["cost_net"].where(claim_costs["declined"], 0))
    step4 = dec.groupby("line").agg(claims=("claim_id", "size"), declined_claims=("declined", "sum"),
                                    declined_incurred=("declined_incurred", "sum"), declined_net=("declined_net", "sum")).reset_index()
    print("Step 4: declined claims removed (cost set to zero, not counted as claims)")
    show(step4, money=["declined_incurred", "declined_net"])

    safe = lambda p: "".join(ch if ch.isalnum() else "_" for ch in str(p).lower())
    paying = claim_costs[~claim_costs["declined"]]
    target_rows, step5_parts = {}, []
    for l in lines:
        rows = read_out(clean_dir3, f"train_{l}")[["row_id", "orig_row_id", "policy_id", "period_start", "period_end", "policy_year", "exposure"]]
        cl = paying[paying["line"] == l]
        orphan = set(cl["row_id"]) - set(rows["row_id"])
        if orphan:
            raise RuntimeError(f"{l}: {len(orphan)} claims point to rows not in the cleaned file")
        totals = cl.groupby("row_id").agg(n_claims=("claim_id", "size"), cost_reported=("cost_target", "sum"), open_estimate=("open_estimate", "sum"))
        by_cost = cl.pivot_table(index="row_id", columns="peril", values="cost_target", aggfunc="sum", fill_value=0)
        by_cost.columns = ["cost_" + safe(c) for c in by_cost.columns]
        by_n = cl.pivot_table(index="row_id", columns="peril", values="claim_id", aggfunc="count", fill_value=0)
        by_n.columns = ["n_" + safe(c) for c in by_n.columns]
        out = rows.merge(totals, on="row_id", how="left").merge(by_cost, on="row_id", how="left").merge(by_n, on="row_id", how="left")
        add_cols = [c for c in out.columns if c not in rows.columns]
        out[add_cols] = out[add_cols].fillna(0)

        # checks: no rows lost or added, money reconciles, nothing negative
        assert len(out) == len(rows) and not out["row_id"].duplicated().any()
        if abs(out["cost_reported"].sum() - cl["cost_target"].sum()) > 1: raise RuntimeError(f"{l}: row totals don't match claim totals")
        if out["n_claims"].sum() != len(cl):                               raise RuntimeError(f"{l}: claim counts don't match")
        if (out["cost_reported"] < 0).any():                               raise RuntimeError(f"{l}: negative row cost")
        if abs(out[list(by_cost.columns)].sum().sum() - out["cost_reported"].sum()) > 1: raise RuntimeError(f"{l}: peril totals don't add up")
        target_rows[l] = out
        out.to_parquet(os.path.join(target_dir, f"target_rows_{l}_step5.parquet"), index=False)

        s = out.groupby("policy_year").agg(rows=("row_id", "size"), rows_with_claim=("n_claims", lambda x: (x > 0).sum()),
                                           policy_years=("exposure", "sum"), claims=("n_claims", "sum"), cost_reported=("cost_reported", "sum")).reset_index()
        step5_parts.append(s.assign(line=l))
    step5 = pd.concat(step5_parts, ignore_index=True)
    step5["claim_freq"] = (step5["claims"] / step5["policy_years"]).round(4)
    step5["cost_per_year"] = step5["cost_reported"] / step5["policy_years"]
    step5["cost_per_claim"] = step5["cost_reported"] / step5["claims"].clip(lower=1)

    print("\nStep 5: reported cost per policy row, by line and policy year")
    show(step5[["line"] + [c for c in step5.columns if c != "line"]], money=["policy_years", "cost_reported", "cost_per_year", "cost_per_claim"])
    print(f"\nReconciled: row totals {fmt(sum(d['cost_reported'].sum() for d in target_rows.values()))} = paying claims {fmt(paying['cost_target'].sum())}")

    claim_costs.to_parquet(os.path.join(target_dir, "claim_costs_step4.parquet"), index=False)
    step5.to_csv(os.path.join(target_dir, "step5_summary.csv"), index=False)
    P3[market]["claim_costs"] = claim_costs


============================== DEVELOPED ==============================
Step 4: declined claims removed (cost set to zero, not counted as claims)
     line  claims  declined_claims declined_incurred declined_net
buildings   26375                0                 0            0
 contents   24052              299                 0            0
    motor  112309             1408                 0            0

Step 5: reported cost per policy row, by line and policy year
     line  policy_year   rows  rows_with_claim policy_years  claims cost_reported  claim_freq cost_per_year cost_per_claim
    motor         2021 216008            24066      187,992 27356.0   105,897,647      0.1455           563          3,871
    motor         2022 254685            25865      199,699 29691.0   116,723,185      0.1487           584          3,931
    motor         2023 254577            24925      199,682 27990.0   117,307,704      0.1402           587          4,191
    motor         2024 254698     

In [23]:
# %% Phase 3 Cell 6 (Step 6): Develop to ultimate ==============================
# Claims are a snapshot at the valuation date, so recent periods are missing
# claims not yet reported. This step MEASURES reporting delays and grosses
# recent periods up to their expected final (ultimate) level.
#
# Method (as Week 1): monthly chain ladder on reporting delay (loss month x
# months until reported), per line and peril; perils with too few claims use
# their line's pooled pattern. Versions: A count pattern, B cost pattern,
# C no development (recent periods get a lower weight instead). A 12-month
# backtest picks the version per line, and each peril's uplift is calibrated by
# how well it predicted late cost, blended towards 1 by credibility.
#
# Emerging market (borrowing): its data only reaches back two years, so
# development beyond its oldest loss month can't be measured there. The
# emerging factor is blended with the developed factor for the same peril:
#     factor = Z x (emerging factor x developed tail) + (1 - Z) x developed factor
#     Z = min(1, sqrt(the peril's own emerging claims / full_cred_claims))
# where the developed tail covers development beyond the emerging data's reach.
# With too little history for its own backtest, the emerging market also takes
# the developed backtest's version choice and per-peril uplift calibration.
#
# Limits (stated in the log): one snapshot, so later changes to estimates on
# claims ALREADY reported can't be measured; open claims are taken at their
# current estimate.

# ---- Settings ---------------------------------------------------------------
min_claims_peril = 300        # fewer claims than this -> use the line's pooled pattern
backtest_months  = 12         # how far back the backtest freezes the data
dev_choice       = "auto"     # "auto" = best backtest per line, or force "A", "B" or "C"
dev_calibrate    = True       # scale each peril's uplift by its backtest accuracy
full_cred_claims = 1082       # claims needed for full credibility (standard for frequency)
k_uplift_bounds  = (0.25, 2)  # limits on the calibration
tail_warn_share  = 0.02       # warn if the oldest year still has this share of cost in open estimates
# -----------------------------------------------------------------------------

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    claim_costs = P3[market]["claim_costs"]
    valuation_date = P3[market]["valuation_date"]
    mi = lambda s: s.dt.year * 12 + s.dt.month - 1        # month index
    V = int(valuation_date.year * 12 + valuation_date.month - 1)

    cl_all = claim_costs[~claim_costs["declined"]].copy()
    cl_all["origin"] = mi(cl_all["loss_date"])
    cl_all["lag"] = (mi(cl_all["report_date"]) - cl_all["origin"]).clip(lower=0)
    cl_all["cnt"] = 1.0

    def cdf_table(d, v, val):
        """Cumulative development factors (age in months -> factor to ultimate) for one group."""
        o_min = int(d["origin"].min()); kmax = v - o_min
        inc = np.zeros((kmax + 1, kmax + 1))
        g = d.groupby(["origin", "lag"])[val].sum()
        for (o, k), x in g.items():
            if o + k <= v:
                inc[o - o_min, k] += x
        C = np.cumsum(inc, axis=1)
        f = np.ones(kmax)
        for j in range(kmax):
            rows = [i for i in range(kmax + 1) if (o_min + i) + j + 1 <= v]
            den = C[rows, j].sum()
            f[j] = C[rows, j + 1].sum() / den if den > 0 else 1.0
        cdf = np.append(np.cumprod(f[::-1])[::-1], 1.0)
        return pd.DataFrame({"age": np.arange(kmax + 1), "cdf": cdf})

    # Which pattern group each claim uses: its own peril, or the pooled line
    peril_groups = cl_all.groupby(["line", "peril"]).size().rename("n").reset_index()
    peril_groups["group"] = np.where(peril_groups["n"] >= min_claims_peril, peril_groups["peril"], "(line pooled)")
    small = peril_groups[peril_groups["group"] == "(line pooled)"]
    if len(small):
        print(f"Perils using their line's pooled pattern (fewer than {min_claims_peril} claims):",
              "; ".join(small["line"] + " " + small["peril"]))
    cl_all = cl_all.merge(peril_groups[["line", "peril", "group"]], on=["line", "peril"])

    def build_cdfs(cl, v):
        out = []
        for (l, grp), d in cl.groupby(["line", "group"]):
            pool = cl[cl["line"] == l] if grp == "(line pooled)" else d
            if len(pool) < min_claims_peril:
                warnings.warn(f"{l} has fewer than {min_claims_peril} claims - no development applied")
                ages = np.arange(v - int(cl["origin"].min()) + 1)
                out.append(pd.DataFrame({"line": l, "group": grp, "age": ages, "cdf_A": 1.0, "cdf_B": 1.0,
                                         "n_pool": len(pool), "kmax": int(ages.max())}))
                continue
            a, b = cdf_table(pool, v, "cnt"), cdf_table(pool, v, "cost_target")
            out.append(pd.DataFrame({"line": l, "group": grp, "age": a["age"], "cdf_A": a["cdf"], "cdf_B": b["cdf"],
                                     "n_pool": len(pool), "kmax": int(a["age"].max())}))
        return pd.concat(out, ignore_index=True)

    # ---- 1. Backtest: freeze the data backtest_months earlier ------------------
    v_bt = V - backtest_months
    backtest, calib = None, pd.DataFrame(columns=["line", "peril", "k_uplift"])
    if v_bt - int(cl_all["origin"].min()) >= backtest_months:
        cl_bt = cl_all[mi(cl_all["report_date"]) <= v_bt]
        cdf_bt = build_cdfs(cl_bt, v_bt)[["line", "group", "age", "cdf_A", "cdf_B"]]
        recent = cl_all[(cl_all["origin"] > v_bt - backtest_months) & (cl_all["origin"] <= v_bt)].copy()
        recent["age_bt"] = v_bt - recent["origin"]
        recent["known_then"] = mi(recent["report_date"]) <= v_bt
        recent["cost_then_c"] = recent["cost_target"].where(recent["known_then"], 0)
        now = cdf_bt.assign(age_bt=cdf_bt["age"] - backtest_months).rename(columns={"cdf_A": "cdf_A_now", "cdf_B": "cdf_B_now"}).drop(columns="age")

        bt = recent.groupby(["line", "group", "origin", "age_bt"]).agg(cost_then=("cost_then_c", "sum"), cost_now=("cost_target", "sum")).reset_index()
        bt = bt.merge(cdf_bt.rename(columns={"age": "age_bt"}), on=["line", "group", "age_bt"], how="left") \
               .merge(now, on=["line", "group", "age_bt"], how="left")
        for c in ["cdf_A", "cdf_B", "cdf_A_now", "cdf_B_now"]:
            bt[c] = bt[c].fillna(1.0)
        bt["pred_A"] = bt["cost_then"] * bt["cdf_A"] / bt["cdf_A_now"]
        bt["pred_B"] = bt["cost_then"] * bt["cdf_B"] / bt["cdf_B_now"]
        bt["pred_C"] = bt["cost_then"]
        backtest = bt.groupby("line").agg(actual_now=("cost_now", "sum"), known_then=("cost_then", "sum"),
                                          pred_A=("pred_A", "sum"), pred_B=("pred_B", "sum"), pred_C=("pred_C", "sum")).reset_index()
        for v_ in "ABC":
            backtest[f"error_{v_}"] = backtest[f"pred_{v_}"] / backtest["actual_now"] - 1
        errs = backtest[["error_A", "error_B", "error_C"]].abs().replace([np.inf, -np.inf], np.nan)
        backtest["best"] = np.where(errs.isna().all(axis=1), "A", errs.fillna(np.inf).idxmin(axis=1).str[-1])

        # Peril-level calibration of the uplift, credibility-weighted
        recent["n_late_c"] = (~recent["known_then"]).astype(int)
        cb = recent.groupby(["line", "peril", "group", "origin", "age_bt"]).agg(
            cost_then=("cost_then_c", "sum"), cost_now=("cost_target", "sum"), n_late=("n_late_c", "sum")).reset_index()
        cb = cb.merge(cdf_bt[["line", "group", "age", "cdf_A"]].rename(columns={"age": "age_bt"}), on=["line", "group", "age_bt"], how="left") \
               .merge(now[["line", "group", "age_bt", "cdf_A_now"]], on=["line", "group", "age_bt"], how="left")
        cb[["cdf_A", "cdf_A_now"]] = cb[["cdf_A", "cdf_A_now"]].fillna(1.0)
        cb["late_pred"] = cb["cost_then"] * cb["cdf_A"] / cb["cdf_A_now"] - cb["cost_then"]
        cb["late_actual"] = cb["cost_now"] - cb["cost_then"]
        calib = cb.groupby(["line", "peril"]).agg(late_actual=("late_actual", "sum"), late_pred=("late_pred", "sum"),
                                                  n_late=("n_late", "sum")).reset_index()
        calib["ratio"] = np.where(calib["late_pred"] > 0, calib["late_actual"] / calib["late_pred"].where(calib["late_pred"] > 0, 1), 1.0)
        calib["z"] = np.minimum(1, np.sqrt(calib["n_late"] / full_cred_claims))
        calib["k_uplift"] = np.clip(calib["z"] * calib["ratio"] + (1 - calib["z"]), *k_uplift_bounds) if dev_calibrate else 1.0
        print("\nUplift calibration per peril (k_uplift multiplies each peril's development uplift)")
        show(calib, money=["late_actual", "late_pred"], dec={"ratio": 3, "z": 3, "k_uplift": 3})
        print(f"\nBacktest: data frozen {backtest_months} months before {valuation_date.date()} - predicting cost reported since then")
        show(backtest, money=["actual_now", "known_then", "pred_A", "pred_B", "pred_C"], pct=["error_A", "error_B", "error_C"])
    else:
        print(f"\nNot enough history for a {backtest_months} month backtest - skipped")
        if borrowing:
            # No backtest of its own: borrow the developed backtest's verdict on the method
            # (which version to use, and how each peril's uplift should be scaled)
            calib = pd.read_csv(os.path.join(borrow_dir, "step6_calibration.csv"))[["line", "peril", "k_uplift"]]
            print("Using the developed market's backtest: its version choice per line and its uplift calibration per peril")

    # ---- 2. Factors on the full data, and the version used per line ------------
    cdfs = build_cdfs(cl_all, V)
    version = {l: ("A" if dev_choice == "auto" else dev_choice) for l in lines}
    if dev_choice == "auto" and backtest is not None:
        version.update(dict(zip(backtest["line"], backtest["best"])))
    elif dev_choice == "auto" and borrowing:
        version.update(pd.read_csv(os.path.join(borrow_dir, "step6_versions.csv")).set_index("line")["version"].to_dict())
    print("\nVersion used per line:", ", ".join(f"{l} = {v}" for l, v in version.items()))

    # line-level count pattern, for the completion weight used by version C
    cdf_line = pd.concat([cdf_table(cl_all[cl_all["line"] == l], V, "cnt").assign(line=l) if (cl_all["line"] == l).any()
                          else pd.DataFrame({"age": np.arange(V - int(cl_all["origin"].min()) + 1), "cdf": 1.0, "line": l})
                          for l in lines], ignore_index=True)

    # Factors actually applied, per line, peril and age (calibrated uplift)
    fac = peril_groups[["line", "peril", "group", "n"]].merge(cdfs, on=["line", "group"]) \
            .merge(calib[["line", "peril", "k_uplift"]], on=["line", "peril"], how="left")
    fac["k_uplift"] = pd.to_numeric(fac["k_uplift"], errors="coerce").fillna(1.0)
    for v_ in "AB":
        fac[f"cdf_{v_}"] = 1 + (fac[f"cdf_{v_}"] - 1) * fac["k_uplift"]

    # ---- 2b. Emerging: blend with the developed factors ------------------------
    if borrowing:
        dev_fac  = pd.read_csv(os.path.join(borrow_dir, "step6_factors_applied.csv"))
        dev_line = pd.read_csv(os.path.join(borrow_dir, "step6_factors_line.csv"))
        def dev_lookup(l, p, ages):
            d = dev_fac[(dev_fac["line"] == l) & (dev_fac["peril"] == p)]
            src = "same peril"
            if d.empty:                                    # peril not in the developed data: its line's pattern
                d = dev_line[dev_line["line"] == l].assign(cdf_A=lambda x: x["cdf"], cdf_B=lambda x: x["cdf"]); src = "developed line"
            d = d.set_index("age")
            top = int(d.index.max())
            get = lambda col: np.array([d.loc[min(int(a), top), col] if a <= top else 1.0 for a in ages])
            return get("cdf_A"), get("cdf_B"), src
        blended, blend_info = [], []
        for (l, p), d in fac.groupby(["line", "peril"]):
            d = d.sort_values("age").copy()
            kmax = int(d["kmax"].iloc[0])
            devA, devB, src = dev_lookup(l, p, d["age"].to_numpy())
            tA, tB, _ = dev_lookup(l, p, np.array([kmax]))           # developed tail beyond the emerging data's reach
            # Credibility rests on the peril's OWN claims: a pooled line pattern says little
            # about a peril that reports differently (e.g. subsidence reports slowly)
            Z = min(1.0, np.sqrt(d["n"].iloc[0] / full_cred_claims))
            d["cdf_A_own"], d["cdf_B_own"] = d["cdf_A"], d["cdf_B"]
            d["cdf_A"] = Z * d["cdf_A_own"] * tA[0] + (1 - Z) * devA
            d["cdf_B"] = Z * d["cdf_B_own"] * tB[0] + (1 - Z) * devB
            d["Z"] = Z
            blended.append(d)
            blend_info.append(dict(line=l, peril=p, peril_claims=int(d["n"].iloc[0]), own_pattern=d["group"].iloc[0] if d["group"].iloc[0] != "(line pooled)" else "line pooled", Z=round(Z, 3),
                                   developed_source=src, developed_tail_A=round(tA[0], 4),
                                   own_A_age0=round(d["cdf_A_own"].iloc[0], 4), blended_A_age0=round(d["cdf_A"].iloc[0], 4)))
        fac = pd.concat(blended, ignore_index=True)
        blend_info = pd.DataFrame(blend_info)
        print("\nEmerging factors blended with the developed market (age 0 = the latest loss month)")
        print(blend_info.to_string(index=False))
        blend_info.to_csv(os.path.join(target_dir, "step6_blend.csv"), index=False)

    # ---- 3. Apply to claims, then to rows ---------------------------------------
    cl_all["age"] = V - cl_all["origin"]
    cl_all = cl_all.drop(columns=[c for c in ["cdf_A", "cdf_B"] if c in cl_all]) \
                   .merge(fac[["line", "peril", "age", "cdf_A", "cdf_B"]], on=["line", "peril", "age"], how="left")
    cl_all[["cdf_A", "cdf_B"]] = cl_all[["cdf_A", "cdf_B"]].fillna(1.0)
    cl_all["cost_dev_A"] = cl_all["cost_target"] * cl_all["cdf_A"]
    cl_all["cost_dev_B"] = cl_all["cost_target"] * cl_all["cdf_B"]

    # Tail check: open estimates in the oldest year are taken as final
    oldest = cl_all[cl_all["origin"] < cl_all["origin"].min() + 12]
    for l, d in oldest.groupby("line"):
        share = d["open_estimate"].sum() / max(d["cost_target"].sum(), 1)
        if share > tail_warn_share:
            print(f"Note: {l} - oldest loss year still has {100 * share:.1f}% of cost in open estimates; "
                  "development beyond the data is not measured (estimates taken as final).")

    tr6, step6_parts = {}, []
    for l in lines:
        rows = read_out(target_dir, f"target_rows_{l}_step5")
        rows = rows.drop(columns=[c for c in ["cost_dev_A", "cost_dev_B", "dev_weight", "cost_target", "dev_version"] if c in rows])
        dev = cl_all[cl_all["line"] == l].groupby("row_id")[["cost_dev_A", "cost_dev_B"]].sum()
        rows = rows.merge(dev, on="row_id", how="left").fillna({"cost_dev_A": 0, "cost_dev_B": 0})
        mid = rows["period_start"] + (rows["period_end"] - rows["period_start"]) / 2
        rows["age_mid"] = V - mi(mid)
        cl_l = cdf_line[cdf_line["line"] == l].set_index("age")["cdf"]
        rows["dev_weight"] = 1 / rows["age_mid"].map(cl_l).fillna(1.0)          # share reported (for version C)
        rows["dev_version"] = version[l]
        rows["cost_target"] = {"A": rows["cost_dev_A"], "B": rows["cost_dev_B"]}.get(version[l], rows["cost_reported"])
        rows = rows.drop(columns="age_mid")
        if (rows["cost_target"] < 0).any():
            raise RuntimeError(f"{l}: negative developed cost")
        rows.to_parquet(os.path.join(target_dir, f"target_rows_{l}_step6.parquet"), index=False)
        tr6[l] = rows
        s = rows.groupby("policy_year").agg(policy_years=("exposure", "sum"), reported=("cost_reported", "sum"), dev_A=("cost_dev_A", "sum"),
                                            dev_B=("cost_dev_B", "sum"), target=("cost_target", "sum"), min_weight=("dev_weight", "min")).reset_index()
        for c in ["reported", "dev_A", "dev_B", "target"]:
            s[c] = s[c] / s["policy_years"]
        s["uplift_target"] = (100 * (s["target"] / s["reported"] - 1)).map(lambda x: f"{x:+.1f}%")
        step6_parts.append(s.assign(line=l))
    step6 = pd.concat(step6_parts, ignore_index=True)

    print("\nCost per policy-year: reported vs developed (version used = target)")
    show(step6[["line"] + [c for c in step6.columns if c != "line"]], money=["policy_years", "reported", "dev_A", "dev_B", "target"], dec={"min_weight": 3})
    show_ages = [a for a in [0, 1, 3, 6, 12, 24] if a <= fac["age"].max()]
    print("\nApplied factors to ultimate (version A) at selected ages (months since loss)")
    print(fac[fac["age"].isin(show_ages)].pivot_table(index=["line", "peril"], columns="age", values="cdf_A").round(4).to_string())

    fac.to_csv(os.path.join(target_dir, "step6_factors_applied.csv"), index=False)
    cdf_line.to_csv(os.path.join(target_dir, "step6_factors_line.csv"), index=False)
    cdfs.to_csv(os.path.join(target_dir, "step6_factors.csv"), index=False)
    pd.DataFrame({"line": list(version), "version": list(version.values())}).to_csv(os.path.join(target_dir, "step6_versions.csv"), index=False)
    step6.to_csv(os.path.join(target_dir, "step6_summary.csv"), index=False)
    if backtest is not None: backtest.to_csv(os.path.join(target_dir, "step6_backtest.csv"), index=False)
    if len(calib): calib.to_csv(os.path.join(target_dir, "step6_calibration.csv"), index=False)
    P3[market]["version"] = version


============================== DEVELOPED ==============================

Uplift calibration per peril (k_uplift multiplies each peril's development uplift)
     line             peril late_actual  late_pred  n_late  ratio     z  k_uplift
buildings accidental_damage      32,932     29,970      25  1.099 0.152     1.015
buildings   escape_of_water     908,453    949,967     171  0.956 0.398     0.983
buildings              fire     698,485    164,020       8  4.259 0.086     1.280
buildings             flood     706,584    746,568      23  0.946 0.146     0.992
buildings             storm     536,709    535,886     124  1.002 0.339     1.001
buildings        subsidence   2,869,352  3,459,274     110  0.829 0.319     0.946
 contents accidental_damage      64,827     69,844      77  0.928 0.267     0.981
 contents   escape_of_water     261,794    262,307      94  0.998 0.295     0.999
 contents             flood     249,089    181,187      22  1.375 0.143     1.053
 contents             t

In [24]:
# %% Phase 3 Cell 7 (Step 7): Trend to the prediction year ======================
# Costs change over time (inflation, risk changes). Developed market (as Week 1):
# the yearly change in cost per policy-year is MEASURED with a Tweedie GLM per
# line, mix-adjusted for the strongest simple rating factors, on cells of rows
# (same estimates as fitting every row, much faster), with still-developing
# periods down-weighted. Sensitivity: other Tweedie powers, and no mix adjustment.
#
# Rule "level_then_project": bring each year to the latest year's level with its
# own mix-adjusted year relativity, then project to the prediction year with the
# fitted annual trend only if year-to-year changes all point the same way.
#
# Emerging market (borrowing): two policy years give one year-to-year change,
# which can't separate trend from noise. Emerging rows are brought to the
# prediction year with the developed market's APPLIED ANNUAL TREND (general
# cost inflation). Developed's year-specific swings (e.g. its 2024 motor drop)
# are not borrowed by default: the emerging data doesn't show them, and the
# brief says costs differ between markets. The emerging GLM is still fitted
# and printed as a check.

# ---- Settings ---------------------------------------------------------------
tweedie_power  = 1.5               # Tweedie power used for the applied trend
power_check    = [1.2, 1.5, 1.8]   # sensitivity check
mix_top_k      = 6                 # number of mix factors to control for
mix_max_levels = 15                # only simple factors (few levels) are candidates
trend_rule     = "level_then_project"   # or "estimate", "if_significant", "none"
emerging_trend = "developed_annual" # emerging: "developed_annual" = developed's applied annual trend (inflation),
                                   # "developed_by_year" = developed's year factors (incl. its own year events), or "own"
# Columns added in Part 2 are kept out of the mix candidates so the developed
# trend is estimated exactly as in Week 1 (and reconciles with it)
mix_exclude    = ["driver_age_band", "policyholder_age_band", "self_employed", "sum_insured_fixed"]
# -----------------------------------------------------------------------------

import statsmodels.api as sm
import statsmodels.formula.api as smf

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    pred_year = int(max(read_out(clean_dir3, f"test_{l}")["policy_year"].max() for l in lines))
    print("Prediction year (from the test files):", pred_year)

    def fit_trend(cells, mix, p, year_as_factor=False):
        rhs = ["C(policy_year)" if year_as_factor else "yr"] + [f"C({m})" for m in mix]
        fam = sm.families.Tweedie(var_power=p, link=sm.families.links.Log())
        return smf.glm("rate ~ " + " + ".join(rhs), data=cells, family=fam, var_weights=cells["w_exp"]).fit(maxiter=100)

    trend_results, year_rel, mix_used = [], [], {}
    for l in lines:
        rows  = read_out(target_dir, f"target_rows_{l}_step6")
        feats = read_out(clean_dir3, f"train_{l}")
        is_cat = lambda s: pd.api.types.is_bool_dtype(s) or pd.api.types.is_string_dtype(s) or s.dtype == object
        cand = [c for c in feats.columns if is_cat(feats[c]) and not c.endswith("_id") and c not in mix_exclude
                and 2 <= feats[c].nunique(dropna=False) <= mix_max_levels]
        d = rows[["row_id", "policy_year", "exposure", "cost_target", "dev_weight"]].merge(feats[["row_id"] + cand], on="row_id", how="left")
        for c in cand:
            d[c] = d[c].astype(str)

        # rank candidates by exposure-weighted spread of cost per policy-year across levels
        overall = d["cost_target"].sum() / d["exposure"].sum()
        def spread(cn):
            g = d.groupby(cn).agg(E=("exposure", "sum"), c=("cost_target", "sum"))
            m = g["c"] / g["E"]
            return (g["E"] * (m - overall) ** 2).sum() / g["E"].sum()
        sp = pd.Series({c: spread(c) for c in cand}).sort_values(ascending=False, kind="mergesort")
        mix = list(sp.index[:mix_top_k]); mix_used[l] = mix

        d["w_e"], d["w_c"] = d["exposure"] * d["dev_weight"], d["cost_target"] * d["dev_weight"]
        cells = d.groupby(["policy_year"] + mix).agg(w_exp=("w_e", "sum"), w_cost=("w_c", "sum")).reset_index()
        cells = cells[cells["w_exp"] > 0].copy()
        cells["rate"] = cells["w_cost"] / cells["w_exp"]
        cells["yr"] = cells["policy_year"] - pred_year

        years = sorted(cells["policy_year"].unique())
        for p in power_check:
            for adj in (True, False):
                if len(years) < 2: break
                m = fit_trend(cells, mix if adj else [], p)
                b, se = m.params["yr"], m.bse["yr"]
                trend_results.append(dict(line=l, power=p, mix_adjusted=adj, trend=np.exp(b) - 1,
                                          lower=np.exp(b - 1.96 * se) - 1, upper=np.exp(b + 1.96 * se) - 1,
                                          converged=bool(getattr(m, "converged", True))))
        print(l, "- mix factors controlled for:", ", ".join(mix))
        if len(years) >= 2:
            mf = fit_trend(cells, mix, tweedie_power, year_as_factor=True)
            rel = [1.0] + [float(np.exp(mf.params[f"C(policy_year)[T.{y}]"])) for y in years[1:]]
        else:
            rel = [1.0]
        year_rel.append(pd.DataFrame({"line": l, "policy_year": years, "relativity": np.round(rel, 4)}))
    trend_results = pd.DataFrame(trend_results)
    year_rel = pd.concat(year_rel, ignore_index=True)
    if not trend_results.empty and not trend_results["converged"].all():
        warnings.warn("Some trend GLMs did not converge - check the table")

    print(f"\nAnnual trend in cost per policy-year (Tweedie GLM, log link), with 95% interval{' - check only, not applied' if borrowing and emerging_trend != 'own' else ''}")
    show(trend_results, pct=["trend", "lower", "upper"])
    print(f"\nYear-by-year relativities to the first year (mix-adjusted, power {tweedie_power})")
    print(year_rel.pivot(index="line", columns="policy_year", values="relativity").to_string())

    # ---- Decide the trend factor for each line and policy year ------------------
    if borrowing and emerging_trend != "own":
        dev_tf = pd.read_csv(os.path.join(borrow_dir, "step7_trend_factors.csv"))
        if int(dev_tf["pred_year"].iloc[0]) != pred_year:
            raise RuntimeError("Developed and emerging prediction years differ - trend factors can't be shared")
        applied = pd.read_csv(os.path.join(borrow_dir, "step7_applied.csv"))
        years_here = sorted(year_rel["policy_year"].unique())
        # Option 1: developed's applied annual trend only (general inflation), no year-specific events
        annual = pd.DataFrame([dict(line=a.line, policy_year=int(y), trend_factor=(1 + a.applied) ** (pred_year - y))
                               for a in applied.itertuples() for y in years_here])
        # Option 2: developed's factors by policy year (its inflation AND its own year-specific swings)
        by_year = dev_tf[dev_tf["policy_year"].isin(years_here)][["line", "policy_year", "trend_factor"]]
        trend_factors = annual if emerging_trend == "developed_annual" else by_year
        cmp = annual.rename(columns={"trend_factor": "annual_trend_only"}).merge(
              by_year.rename(columns={"trend_factor": "developed_year_factors"}), on=["line", "policy_year"])
        cmp = cmp.merge(year_rel.rename(columns={"relativity": "emerging_own_relativity"}), on=["line", "policy_year"])
        print(f"\nEmerging: trend borrowed from the developed market ({emerging_trend} applied)")
        print("Year-specific developed swings are only borrowed under 'developed_by_year'; compare with the emerging data's own year pattern:")
        show(cmp, dec={"annual_trend_only": 4, "developed_year_factors": 4, "emerging_own_relativity": 4})
    else:
        cons = []
        for l, g in year_rel.sort_values("policy_year").groupby("line"):
            ch = np.diff(np.log(g["relativity"].to_numpy()))
            cons.append(dict(line=l, latest_year=int(g["policy_year"].max()), consistent=len(ch) > 0 and (all(ch > 0) or all(ch < 0))))
        applied = trend_results[(trend_results["power"] == tweedie_power) & trend_results["mix_adjusted"]].merge(pd.DataFrame(cons), on="line")
        applied["applied"] = np.select(
            [trend_rule == "none",
             (trend_rule == "if_significant") & (applied["lower"] <= 0) & (applied["upper"] >= 0),
             (trend_rule == "level_then_project") & ~applied["consistent"]],
            [0.0, 0.0, 0.0], applied["trend"])
        tf = []
        for _, a in applied.iterrows():
            rel = year_rel[year_rel["line"] == a["line"]].set_index("policy_year")["relativity"]
            for y in rel.index:
                f = (rel[a["latest_year"]] / rel[y]) * (1 + a["applied"]) ** (pred_year - a["latest_year"]) \
                    if trend_rule == "level_then_project" else (1 + a["applied"]) ** (pred_year - y)
                tf.append(dict(line=a["line"], policy_year=int(y), trend_factor=f, pred_year=pred_year))
        trend_factors = pd.DataFrame(tf)
        print(f"\nRule: {trend_rule}. Projection to {pred_year} used only where year-to-year changes are consistent:")
        show(applied[["line", "latest_year", "consistent", "trend", "applied"]], pct=["trend", "applied"])

    step7 = []
    for l in lines:
        rows = read_out(target_dir, f"target_rows_{l}_step6")
        rows = rows.drop(columns=[c for c in ["trend_factor"] if c in rows]) \
                   .merge(trend_factors[trend_factors["line"] == l][["policy_year", "trend_factor"]], on="policy_year", how="left")
        if rows["trend_factor"].isna().any():
            raise RuntimeError(f"{l}: no trend factor for policy years {sorted(rows.loc[rows['trend_factor'].isna(), 'policy_year'].unique())}")
        rows["cost_trended"] = rows["cost_target"] * rows["trend_factor"]
        if not np.isfinite(rows["cost_trended"]).all() or (rows["cost_trended"] < 0).any():
            raise RuntimeError(f"{l}: invalid trended cost")
        rows.to_parquet(os.path.join(target_dir, f"target_rows_{l}_step7.parquet"), index=False)
        s = rows.groupby("policy_year").agg(policy_years=("exposure", "sum"), before=("cost_target", "sum"),
                                            trend_factor=("trend_factor", "first"), after=("cost_trended", "sum")).reset_index()
        s["before"] /= s["policy_years"]; s["after"] /= s["policy_years"]
        step7.append(s.assign(line=l))
    step7 = pd.concat(step7, ignore_index=True)
    print(f"\nCost per policy-year before and after trending to {pred_year}")
    show(step7[["line", "policy_year", "policy_years", "before", "trend_factor", "after"]], money=["policy_years", "before", "after"], dec={"trend_factor": 4})

    trend_results.to_csv(os.path.join(target_dir, "step7_trend.csv"), index=False)
    year_rel.to_csv(os.path.join(target_dir, "step7_year_relativities.csv"), index=False)
    applied.to_csv(os.path.join(target_dir, "step7_applied.csv"), index=False)
    trend_factors.assign(pred_year=pred_year).to_csv(os.path.join(target_dir, "step7_trend_factors.csv"), index=False)
    step7.to_csv(os.path.join(target_dir, "step7_summary.csv"), index=False)
    P3[market]["applied"] = applied
    P3[market]["pred_year"] = pred_year


============================== DEVELOPED ==============================
Prediction year (from the test files): 2025
motor - mix factors controlled for: marital_status, urban_density, vehicle_use, region, ncb_capped, new_business
buildings - mix factors controlled for: flood_zone_missing, energy_rating, property_type, region, occupancy, construction_type
contents - mix factors controlled for: tenure, region, flood_zone_missing, urban_density, credit_band, payment_method

Annual trend in cost per policy-year (Tweedie GLM, log link), with 95% interval
     line  power  mix_adjusted  trend  lower   upper  converged
    motor    1.2          True -3.18% -6.11%  -0.17%       True
    motor    1.2         False -2.53% -6.64%  +1.76%       True
    motor    1.5          True -3.26% -6.03%  -0.41%       True
    motor    1.5         False -2.56% -6.68%  +1.74%       True
    motor    1.8          True -3.31% -5.94%  -0.61%       True
    motor    1.8         False -2.59% -6.71%  +1.71%       T

In [25]:
# %% Phase 3 Cell 8 (Step 8): Large claims - cap and spread back ===============
# A handful of very large losses can make the model think whole groups are
# riskier than they are. Each row's cost is capped at a quantile of positive
# row costs (per line, set from the data), and the amount above the cap is
# spread back across the line as a loading, so no money is lost:
#   target for the model = min(row cost, cap)
#   large_loading        = total cost / total capped cost   (per line)
#   final prediction     = model prediction x large_loading (applied in Phase 4)
#
# Emerging market (borrowing): with ~1,000 property claims, the loading rests on
# a few very large rows. It is blended with the developed loading by credibility:
#   loading = Z x own loading + (1 - Z) x developed loading,
#   Z = min(1, sqrt(rows with a cost / full_cred_large))
# Large-claim severity needs far more data than claim frequency, hence a higher
# full-credibility standard than Step 6.

# ---- Settings ---------------------------------------------------------------
cap_large       = True    # False = no capping (loading = 1)
cap_quantile    = 0.995   # cap = this quantile of positive row costs, per line
full_cred_large = 5000    # rows with a cost needed to trust a line's own loading fully
# -----------------------------------------------------------------------------

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    dev_large = pd.read_csv(os.path.join(borrow_dir, "step8_large_claims.csv")).set_index("line") if borrowing else None
    step8, large_by_year = [], []
    for l in lines:
        rows = read_out(target_dir, f"target_rows_{l}_step7")
        if "cost_trended" not in rows:
            raise RuntimeError(f"{l}: run Cell 7 first")
        pos = rows.loc[rows["cost_trended"] > 0, "cost_trended"]
        cap = float(np.quantile(pos, cap_quantile)) if (cap_large and len(pos)) else np.inf
        rows["cost_capped"] = np.minimum(rows["cost_trended"], cap)
        rows["above_cap"] = rows["cost_trended"] - rows["cost_capped"]
        own = rows["cost_trended"].sum() / rows["cost_capped"].sum() if rows["cost_capped"].sum() > 0 else 1.0
        if abs(rows["cost_capped"].sum() * own - rows["cost_trended"].sum()) > 1: raise RuntimeError(f"{l}: loading doesn't restore the total")
        if (rows["cost_capped"] < 0).any(): raise RuntimeError(f"{l}: negative capped cost")

        if borrowing:
            z = min(1.0, np.sqrt(len(pos) / full_cred_large))
            loading = z * own + (1 - z) * float(dev_large.loc[l, "own_loading"])
        else:
            z, loading = 1.0, own
        rows["own_loading"], rows["large_loading"] = own, loading
        rows.to_parquet(os.path.join(target_dir, f"target_rows_{l}_step8.parquet"), index=False)

        step8.append(dict(line=l, cap=cap, rows_capped=int((rows["above_cap"] > 0).sum()),
                          share_of_rows_with_cost=(rows["above_cap"] > 0).sum() / max(len(pos), 1),
                          cost_above_cap=rows["above_cap"].sum(), share_of_cost_above_cap=rows["above_cap"].sum() / rows["cost_trended"].sum(),
                          own_loading=own, Z=z, developed_loading=float(dev_large.loc[l, "own_loading"]) if borrowing else np.nan,
                          large_loading=loading, largest_row=rows["cost_trended"].max()))
        yb = rows.groupby("policy_year").agg(rows_capped=("above_cap", lambda x: (x > 0).sum()), above=("above_cap", "sum"), tot=("cost_trended", "sum")).reset_index()
        yb["share_above_cap"] = (100 * yb["above"] / yb["tot"]).map(lambda x: f"{x:.2f}%")
        large_by_year.append(yb.drop(columns=["above", "tot"]).assign(line=l))
    step8 = pd.DataFrame(step8); large_by_year = pd.concat(large_by_year, ignore_index=True)

    print(f"Step 8: large claim cap per line (cap = {cap_quantile} quantile of positive row costs)")
    show(step8, money=["cap", "cost_above_cap", "largest_row"], pct=["share_of_rows_with_cost", "share_of_cost_above_cap"],
         dec={"own_loading": 4, "developed_loading": 4, "large_loading": 4, "Z": 3})
    print("\nShare of cost above the cap, by year (should look random, not trending)")
    show(large_by_year)

    step8.to_csv(os.path.join(target_dir, "step8_large_claims.csv"), index=False)
    large_by_year.to_csv(os.path.join(target_dir, "step8_large_by_year.csv"), index=False)
    P3[market]["step8"] = step8


============================== DEVELOPED ==============================
Step 8: large claim cap per line (cap = 0.995 quantile of positive row costs)
     line     cap  rows_capped share_of_rows_with_cost cost_above_cap share_of_cost_above_cap  own_loading   Z  developed_loading  large_loading largest_row
    motor  54,699          491                  +0.50%     35,111,723                  +8.69%       1.0952 1.0                NaN         1.0952  10,756,480
buildings 194,239          127                  +0.50%     13,906,808                  +4.38%       1.0458 1.0                NaN         1.0458     784,019
 contents  34,128          115                  +0.50%      1,541,534                  +2.05%       1.0209 1.0                NaN         1.0209     122,691

Share of cost above the cap, by year (should look random, not trending)
 policy_year  rows_capped share_above_cap      line
        2021          130           5.92%     motor
        2022          141           6.45%   

In [26]:
# %% Phase 3 Cell 9 (Step 9): Final checks, the final target, reconciliation ===
# Re-checks every step from the saved files, reconciles the money from reported
# cost to the final target, saves one target file per line for Phase 4 and a
# record of this run's choices. For the developed market, checks the Python
# port reproduces the Week 1 R choices.

for market in markets:
    print(f"\n{'=' * 30} {market.upper()} {'=' * 30}")
    prep_dir3, clean_dir3, target_dir, borrowing = (P3[market][k] for k in ("prep_dir", "clean_dir", "target_dir", "borrowing"))
    # Results from Cells 2-8: taken from memory if this session still has them,
    # otherwise re-read from the files those cells saved (so this cell can be rerun alone)
    if all(k in P3[market] for k in ("valuation_date", "version", "applied", "pred_year", "step8")):
        valuation_date, version, applied, pred_year, step8 = (P3[market][k] for k in
            ("valuation_date", "version", "applied", "pred_year", "step8"))
    else:
        print("  (re-reading Cell 2-8 results from saved files)")
        valuation_date = read_out(target_dir, "claim_costs_step4")["report_date"].max()
        version   = {l: str(read_out(target_dir, f"target_rows_{l}_step6")["dev_version"].iloc[0]) for l in lines}
        applied   = pd.read_csv(os.path.join(target_dir, "step7_applied.csv"))
        pred_year = int(pd.read_csv(os.path.join(target_dir, "step7_trend_factors.csv"))["pred_year"].iloc[0])
        step8     = pd.read_csv(os.path.join(target_dir, "step8_large_claims.csv"))
    final_summary, recon = [], []
    for l in lines:
        s5, s6, s7, s8 = (read_out(target_dir, f"target_rows_{l}_step{k}") for k in (5, 6, 7, 8))
        tr = read_out(clean_dir3, f"train_{l}")
        for x in (s5, s6, s7, s8):
            if set(x["row_id"]) != set(tr["row_id"]) or x["row_id"].duplicated().any():
                raise RuntimeError(f"{l}: target rows don't match the cleaned training rows")
        for c in ["cost_reported", "cost_target", "cost_trended", "cost_capped"]:
            if not np.isfinite(s8[c]).all() or (s8[c] < 0).any():
                raise RuntimeError(f"{l}: invalid values in {c}")
        if (s8["exposure"] <= 0).any():                                          raise RuntimeError(f"{l}: rows with zero or negative exposure")
        if abs(s5["cost_reported"].sum() - s8["cost_reported"].sum()) > 1:      raise RuntimeError(f"{l}: reported cost changed after Step 5")
        if abs(s6["cost_target"].sum() - s8["cost_target"].sum()) > 1:          raise RuntimeError(f"{l}: developed cost changed after Step 6")
        if abs(s7["cost_trended"].sum() - s8["cost_trended"].sum()) > 1:        raise RuntimeError(f"{l}: trended cost changed after Step 7")
        if abs(s8["cost_capped"].sum() * s8["own_loading"].iloc[0] - s8["cost_trended"].sum()) > 1: raise RuntimeError(f"{l}: loading doesn't restore the total")

        recon.append(dict(line=l, reported=s8["cost_reported"].sum(), developed=s8["cost_target"].sum(), trended=s8["cost_trended"].sum(),
                          capped=s8["cost_capped"].sum(), capped_x_own_loading=s8["cost_capped"].sum() * s8["own_loading"].iloc[0],
                          capped_x_applied_loading=s8["cost_capped"].sum() * s8["large_loading"].iloc[0]))
        final = s8[["row_id", "orig_row_id", "policy_id", "policy_year", "exposure"]].assign(
            target=s8["cost_capped"],              # what the model learns
            target_uncapped=s8["cost_trended"],    # developed + trended, before the cap
            large_loading=s8["large_loading"],     # multiply predictions by this
            dev_weight=s8["dev_weight"], dev_version=s8["dev_version"], n_claims=s8["n_claims"])
        final.to_parquet(os.path.join(target_dir, f"target_final_{l}.parquet"), index=False)
        fs = final.groupby("policy_year").agg(policy_years=("exposure", "sum"), rows=("row_id", "size"),
                                              share_rows_with_claim=("n_claims", lambda x: (x > 0).mean()),
                                              target=("target", "sum"), loading=("large_loading", "first")).reset_index()
        fs["target_per_year"] = fs["target"] / fs["policy_years"]
        fs["with_loading_per_year"] = fs["target_per_year"] * fs["loading"]
        final_summary.append(fs.drop(columns=["target", "loading"]).assign(line=l))
    recon = pd.DataFrame(recon); final_summary = pd.concat(final_summary, ignore_index=True)

    print("Money trail from reported cost to the final target (all rows, all years)")
    show(recon, money=[c for c in recon.columns if c != "line"])
    print(f"\nFinal target per policy-year, by line and year ({pred_year} cost level)")
    show(final_summary[["line"] + [c for c in final_summary.columns if c != "line"]], money=["policy_years", "target_per_year", "with_loading_per_year"], dec={"share_rows_with_claim": 4})

    run_record = {
        "market": market, "valuation_date": str(valuation_date.date()), "prediction_year": pred_year,
        "development_versions": "; ".join(f"{k}={v}" for k, v in version.items()),
        "development_calibrated": globals().get("dev_calibrate", "see Cell 6 settings"),
        "development_borrowed": "blended with developed by credibility (see step6_blend.csv)" if borrowing else "no (source market)",
        "trend_rule": f"borrowed from developed ({globals().get('emerging_trend', 'developed_annual')})" if (borrowing and globals().get("emerging_trend", "developed_annual") != "own")
                      else globals().get("trend_rule", "see Cell 7 settings"),
        "trend_applied": "; ".join(f"{r.line}={r.applied:+.4f}" for r in applied.itertuples()),
        "cap_quantile": (globals().get("cap_quantile") if globals().get("cap_large", True) else None) or "see Cell 8 settings",
        "large_loading": "; ".join(f"{r.line}={r.large_loading:.4f} (Z={r.Z:.3f})" for r in step8.itertuples()),
    }
    with open(os.path.join(target_dir, "target_run_record.txt"), "w") as fh:
        fh.write("\n".join(f"{k}: {v}" for k, v in run_record.items()))
    recon.to_csv(os.path.join(target_dir, "target_money_trail.csv"), index=False)
    final_summary.to_csv(os.path.join(target_dir, "target_final_summary.csv"), index=False)
    print("\n" + "\n".join(f"{k}: {v}" for k, v in run_record.items()))

    # ---- Reconcile with the Week 1 R run (developed only) ------------------------
    week1_r = {"version": {"motor": "A", "buildings": "A", "contents": "A"},
               "trend_applied": {"motor": 0.0, "buildings": 0.0865, "contents": 0.0522}}
    if market == source_market:
        chk = pd.DataFrame({"week1_R_version": week1_r["version"], "python_version": version,
                            "week1_R_trend": week1_r["trend_applied"],
                            "python_trend": applied.set_index("line")["applied"].round(4).to_dict()})
        chk["match"] = (chk["week1_R_version"] == chk["python_version"]) & ((chk["week1_R_trend"] - chk["python_trend"]).abs() < 0.0002)
        print("\n", chk.to_string())
        if not chk["match"].all():
            raise RuntimeError("Python Phase 3 does not reproduce the Week 1 R choices - investigate before continuing")
        print("OK: Python Phase 3 reproduces the Week 1 R development versions and trends")
    print(f"\nPhase 3 complete for {market}. Final targets: target_final_<line>.parquet in {target_dir}")

# Phase 4 reads the saved target files, so the in-memory tables can go
for m in P3:
    for k in [k for k in P3[m] if k not in ("prep_dir", "clean_dir", "target_dir", "borrowing")]:
        del P3[m][k]
gc.collect()
print("\nPhase 3 tables cleared from memory (all outputs are saved to disk).")


============================== DEVELOPED ==============================
Money trail from reported cost to the final target (all rows, all years)
     line    reported   developed     trended      capped capped_x_own_loading capped_x_applied_loading
    motor 430,595,512 444,480,646 404,095,480 368,983,758          404,095,480              404,095,480
buildings 247,768,586 255,022,201 317,539,438 303,632,629          317,539,438              317,539,438
 contents  65,966,579  66,653,295  75,262,499  73,720,964           75,262,499               75,262,499

Final target per policy-year, by line and year (2025 cost level)
     line  policy_year policy_years   rows  share_rows_with_claim target_per_year with_loading_per_year
    motor         2021      187,992 216008                 0.1114             475                   521
    motor         2022      199,699 254685                 0.1016             478                   524
    motor         2023      199,682 254577                 0

Phase 4

In [27]:
# =============================================================================
# PART 2 - PHASE 4: BAYESIAN CREDIBILITY GLM, VALIDATION AND SUBMISSION
#
# Goal: predict each emerging 2025 row's expected cost (scored with Tweedie D^2,
# power 1.5). The emerging market is small, so the model borrows from the
# developed market using Bayesian (empirical Bayes) credibility:
#
#   1. Developed model: a Tweedie GLM per line on the developed market, using
#      only features BOTH markets have. This is the PRIOR for emerging.
#   2. Emerging correction: a second GLM fitted on emerging data that starts
#      from the developed prediction and learns only where emerging differs.
#      Each correction is pulled towards zero (= "same as developed") by a
#      penalty. Bayes: prior N(0, tau^2) on each correction  <=>  penalty phi/tau^2.
#      Emerging-only features (region E01-E06, gender, ...) get their own prior.
#   3. tau^2 (how much the markets truly differ) is ESTIMATED from the data:
#      compare emerging vs developed estimates factor by factor; the spread left
#      after removing emerging noise is the real difference.
#      Credibility per coefficient: Z = tau^2 / (tau^2 + its emerging noise^2).
#   4. Validation on the latest emerging year, across a range of tau^2 that runs
#      from 0 ("developed relativities, emerging level only") to infinity
#      ("emerging data only") - the two benchmark options are the ends of the scale.
#   5. Final fit on all emerging years, predict the test rows, check, submit.
#
# The emerging price LEVEL (intercept) is never shrunk: costs differ by market.
# Nothing is typed in for this dataset; every band, level and weight comes
# from the data.
# =============================================================================

# %% Phase 4 Cell 1: Settings and functions ====================================

# ---- SETTINGS ---------------------------------------------------------------
source_market  = "developed"          # the prior market
target_market  = "emerging"           # the market we predict
lines          = ["motor", "buildings", "contents"]
tweedie_p      = 1.5                  # = the competition metric's power
n_bands        = 4                    # bands for continuous features (quantiles of the developed data)
max_as_factor  = 12                   # numeric columns with this many values or fewer are categories
min_level_expo, min_level_claims = 500, 50     # developed: thinner levels join "other"
min_level_expo_new, min_level_claims_new = 200, 20   # emerging-only features (smaller market)
max_levels     = 10                   # at most this many levels per feature
money_features = ["sum_insured", "written_premium"]   # used relative to their market's median that year
drop_features  = ["customer_since_year", "year_built",      # duplicated by customer_tenure / property_age
                  "customer_service_contacts",              # leakage: caused by claims (Week 1 check)
                  "weather_imputed", "sum_insured_fixed"]   # bookkeeping flags, not risk
dup_cor        = 0.98                 # near-duplicate numeric features: keep the first
tau_grid       = [0, 0.25, 0.5, 1, 2, 4, float("inf")]      # multiples of the estimated tau^2 tried on the holdout
tau_grid_new   = [0, 0.25, 1, 4]       # multiples of the emerging-only features' tau^2 (region, gender) tried on the holdout
tau_choice     = "holdout"            # "holdout" = best multiple on the holdout year, or "eb" = estimated tau^2 as is
own_level_features = ["new_business"] # candidates NOT shrunk to developed (they define the 2025 mix shift):
                                      # Cell 3 tests the model with and without this and keeps what scores better
min_code_overlap = 0.5                # a shared column whose emerging values mostly don't exist in developed
                                      # (e.g. region E01-E06 vs R01-R12) is treated as emerging-only
weak_prior_sd  = 1.0                  # weak prior on every log-relativity (keeps thin levels from exploding)
max_relativity = 20                   # review gate: flag any final relativity beyond 20x (or below 1/20)
reviewed_ok    = False                # set True after a person has reviewed flagged results
# -----------------------------------------------------------------------------

import os, re, glob, gc, json, pickle, warnings
import numpy as np
import pandas as pd

work_dir  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
model_dir = os.path.join(work_dir, "model")
os.makedirs(model_dir, exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_rows", 500)
for m in (source_market, target_market):
    for d in (f"clean_{m}", f"target_{m}"):
        if not os.path.isdir(os.path.join(work_dir, d)):
            raise RuntimeError(f"Missing {d} - run Phases 1-3 first")

fmt = lambda x, d=0: f"{x:,.{d}f}"
def show(df, money=(), pct=(), dec=None):
    out = df.copy()
    for c in money:
        if c in out: out[c] = out[c].map(lambda x: fmt(x) if pd.notna(x) else "")
    for c in pct:
        if c in out: out[c] = out[c].map(lambda x: f"{100 * x:+.2f}%" if pd.notna(x) else "")
    if dec: out = out.round(dec)
    print(out.to_string(index=False))

# ---- Metric -------------------------------------------------------------------
def tw_dev(y, mu, p=tweedie_p):
    mu = np.maximum(mu, 1e-10)
    t1 = np.where(y > 0, np.power(np.maximum(y, 0), 2 - p) / ((1 - p) * (2 - p)), 0.0)
    return 2 * np.sum(t1 - y * mu ** (1 - p) / (1 - p) + mu ** (2 - p) / (2 - p))

def d2(y, mu, expo):
    """Tweedie D^2 as the metric: 1 - deviance / deviance of a constant rate."""
    return 1 - tw_dev(y, mu) / tw_dev(y, y.sum() / expo.sum() * expo)

def c_star(y, mu, p=tweedie_p):
    """The single multiplier that would minimise deviance: 1 = level right."""
    return np.sum(y * mu ** (1 - p)) / np.sum(mu ** (2 - p))

# ---- Data ---------------------------------------------------------------------
def load_line(market, line):
    """Training rows (with target) and test rows for one market and line."""
    tgt = pd.read_parquet(os.path.join(work_dir, f"target_{market}", f"target_final_{line}.parquet"))
    tr  = pd.read_parquet(os.path.join(work_dir, f"clean_{market}", f"train_{line}.parquet"))
    te  = pd.read_parquet(os.path.join(work_dir, f"clean_{market}", f"test_{line}.parquet"))
    tr  = tr.drop(columns=[c for c in ["exposure", "policy_year", "orig_row_id", "policy_id"] if c in tr])
    train = tgt[["row_id", "orig_row_id", "policy_year", "exposure", "target", "target_uncapped", "large_loading"]] \
              .merge(tr, on="row_id", how="inner")
    assert len(train) == len(tgt), f"{market} {line}: target and cleaned rows don't match"
    return train, te

def feature_cols(d):
    excl = {"row_id", "orig_row_id", "policy_id", "exposure", "policy_year", "weather_year", "exposure_kaggle",
            "target", "target_uncapped", "large_loading", *drop_features}
    return [c for c in d.columns if c not in excl and not c.endswith("_id")
            and not pd.api.types.is_datetime64_any_dtype(d[c])]

def money_relative(d):
    """Money features relative to their market's median in the same policy year
    (removes inflation and the difference in price level between markets)."""
    d = d.copy()
    for c in money_features:
        if c in d:
            d[c] = d[c] / d.groupby("policy_year")[c].transform("median")
    return d

# ---- Feature preparation: bands and levels, learned from fit data only -------
def make_spec(fit, feats, min_expo, min_claims):
    spec = {}
    fit = money_relative(fit)
    num = [f for f in feats if pd.api.types.is_numeric_dtype(fit[f]) and not pd.api.types.is_bool_dtype(fit[f])
           and fit[f].nunique() > 2]
    drop_dup = []
    if len(num) > 1:
        sample = fit[num].sample(min(len(fit), 200000), random_state=0)
        cm = sample.corr().abs().fillna(0).to_numpy()
        for i in range(1, len(num)):
            for j in range(i):
                if num[j] not in drop_dup and cm[i, j] > dup_cor:
                    drop_dup.append(num[i]); break
    for f in feats:
        if f in drop_dup:
            continue
        x = fit[f]
        if pd.api.types.is_numeric_dtype(x) and not pd.api.types.is_bool_dtype(x) and x.nunique() > max_as_factor:
            br = np.unique(np.quantile(x, np.linspace(0, 1, n_bands + 1)[1:-1]))
            kind, lv = "band", band(x, br)
        else:
            kind, br, lv = "cat", None, x.astype(str)
        g = pd.DataFrame({"lv": lv.values, "e": fit["exposure"].values, "c": (fit["target"] > 0).values}) \
              .groupby("lv").agg(e=("e", "sum"), c=("c", "sum"))
        g = g[(g["e"] >= min_expo) & (g["c"] >= min_claims)].sort_values("e", ascending=False)
        kept = list(g.index[:max_levels])
        if len(kept) < 2:
            continue
        spec[f] = {"kind": kind, "breaks": br, "levels": kept, "base": kept[0],
                   "has_other": bool((~lv.isin(kept)).any())}
    return spec, drop_dup

def band(x, br):
    return pd.Series(np.searchsorted(br, np.asarray(x, dtype=float), side="right"), index=x.index).map(lambda k: f"b{k}")

def design(d, spec):
    """Dummy columns (base level left out) for every feature in spec."""
    d = money_relative(d)
    cols, names = [], []
    for f, s in spec.items():
        lv = band(d[f], s["breaks"]) if s["kind"] == "band" else d[f].astype(str)
        fallback = "other" if s["has_other"] else s["base"]
        lv = lv.where(lv.isin(s["levels"]), fallback)
        for level in [l for l in s["levels"] if l != s["base"]] + (["other"] if s["has_other"] else []):
            cols.append((lv.values == level).astype(np.float32)); names.append(f"{f}={level}")
    X = np.column_stack(cols) if cols else np.zeros((len(d), 0), np.float32)
    return X, names

# ---- Tweedie GLM by (penalised) IRLS ------------------------------------------
def fit_tweedie(X, y, offset, penalty=None, prior_sd=None, max_iter=50, tol=1e-9, label=""):
    """Tweedie GLM, log link, with a ridge penalty per coefficient (column 0 =
    intercept, never penalised). Bayes: penalty = phi / prior variance.
    penalty: fixed values; prior_sd: a weak N(0, prior_sd^2) prior whose penalty
    phi / prior_sd^2 is updated with phi each iteration.
    Returns coefficients, their covariance (phi x inverse information) and phi."""
    n, k = X.shape
    Xi = np.column_stack([np.ones(n, np.float32), X]).astype(np.float32)
    pen = np.zeros(k + 1) if penalty is None else np.concatenate([[0.0], penalty])
    beta = np.zeros(k + 1); beta[0] = np.log(max(y.sum(), 1e-9) / np.exp(offset).sum())
    def objective(bt, pen):
        """Penalised deviance: what IRLS minimises (deviance/2 + penalty/2)."""
        return tw_dev(y, np.exp(offset + Xi @ bt)) / 2 + 0.5 * np.sum(pen * bt ** 2)
    obj_old, converged, it = np.inf, False, 0
    for it in range(max_iter):
        eta = offset + Xi @ beta
        mu = np.exp(eta)
        if prior_sd is not None and it < 8:                # phi settles in a few iterations, then is held fixed
            phi = np.sum((y - mu) ** 2 / mu ** tweedie_p) / max(n - (k + 1), 1)
            pen = np.concatenate([[0.0], np.full(k, phi / prior_sd ** 2)])
            obj_old = np.inf
        w = mu ** (2 - tweedie_p)
        z = (eta - offset) + (y - mu) / mu
        A = np.zeros((k + 1, k + 1)); b = np.zeros(k + 1)
        for s in range(0, n, 200000):                     # in chunks to save memory
            Xc, wc = Xi[s:s + 200000].astype(np.float64), w[s:s + 200000]
            A += (Xc * wc[:, None]).T @ Xc
            b += Xc.T @ (wc * z[s:s + 200000])
        A[np.diag_indices_from(A)] += pen + 1e-8
        step = np.linalg.solve(A, b) - beta
        # step-halving: never accept a step that makes the fit worse (heavy-tailed data can oscillate)
        obj_now, t = objective(beta, pen), 1.0
        while t > 1e-4 and objective(beta + t * step, pen) > obj_now + 1e-12 * abs(obj_now):
            t /= 2
        beta = beta + t * step
        obj = objective(beta, pen)
        if abs(obj_old - obj) <= tol * max(abs(obj), 1e-9) or t <= 1e-4:
            converged = True; break
        obj_old = obj
    mu = np.exp(offset + Xi @ beta)
    phi = np.sum((y - mu) ** 2 / mu ** tweedie_p) / max(n - (k + 1), 1)
    cov = phi * np.linalg.inv(A)
    if label:
        print(f"  {label}: {n:,} rows x {k} coefficients, {it + 1} iterations, converged: {converged}")
    return {"beta": beta, "cov": cov, "phi": phi, "converged": converged}

def lin_pred(X, beta):
    """Linear predictor without the intercept."""
    return X @ beta[1:] if X.shape[1] else np.zeros(len(X))

print("Phase 4 setup done. Prior market:", source_market, "| predicting:", target_market, "| model folder:", model_dir)

Phase 4 setup done. Prior market: developed | predicting: emerging | model folder: /kaggle/working/model


In [28]:
# %% Phase 4 Cell 2: The prior - developed model per line ======================
# A Tweedie GLM per line on ALL developed years, using only features both markets
# have (exact ages, telematics, mileage, credit band exist only in developed and
# are left out). Saved to model/prior_<line>.pkl for Cells 3 and 4.

prior_info = []
for l in lines:
    print(f"\n==> {l}")
    dev, _ = load_line(source_market, l)
    em, em_test = load_line(target_market, l)
    f_dev, f_em = feature_cols(dev), feature_cols(em)
    shared   = [f for f in f_dev if f in f_em]
    dev_only = [f for f in f_dev if f not in f_em]
    em_only  = [f for f in f_em if f not in f_dev]
    # Same column, different codes (e.g. region): emerging values that developed never uses
    # can't borrow a developed effect, so the column is modelled as emerging-only
    for f in list(shared):
        if pd.api.types.is_numeric_dtype(em[f]) and not pd.api.types.is_bool_dtype(em[f]) and em[f].nunique() > max_as_factor:
            continue
        overlap = em[f].astype(str).isin(set(dev[f].astype(str))).mean()
        if overlap < min_code_overlap:
            shared.remove(f); em_only.append(f)
            print(f"  {f}: only {overlap:.0%} of emerging values exist in developed -> emerging-only")
    print(f"  shared features: {len(shared)} | developed only (not used): {', '.join(dev_only) or 'none'}")
    print(f"  emerging only (used in the emerging correction): {', '.join(em_only) or 'none'}")

    spec, dups = make_spec(dev, shared, min_level_expo, min_level_claims)
    if dups: print("  dropped as near-duplicates:", ", ".join(dups))
    Xd, names = design(dev, spec)
    fit = fit_tweedie(Xd, dev["target"].to_numpy(float), np.log(dev["exposure"].to_numpy(float)), prior_sd=weak_prior_sd, label="developed GLM")
    rel = pd.Series(np.exp(fit["beta"][1:]), index=names)
    big = rel.iloc[np.argsort(-np.abs(np.log(rel.to_numpy())))][:6]
    print("  largest relativities:", "; ".join(f"{k} = {v:.2f}" for k, v in big.items()))
    with open(os.path.join(model_dir, f"prior_{l}.pkl"), "wb") as fh:
        pickle.dump({"spec": spec, "names": names, "beta": fit["beta"], "cov": fit["cov"],
                     "shared": shared, "em_only": em_only}, fh)
    prior_info.append(dict(line=l, rows=len(dev), shared_features=len(spec), coefficients=len(names),
                           converged=fit["converged"], max_relativity=float(np.exp(np.abs(fit["beta"][1:]).max())) if len(names) else 1.0))
    del dev, em, em_test, Xd; gc.collect()

prior_info = pd.DataFrame(prior_info)
print("\nPrior (developed) models"); show(prior_info, dec={"max_relativity": 2})
prior_info.to_csv(os.path.join(model_dir, "prior_models.csv"), index=False)


==> motor
  region: only 0% of emerging values exist in developed -> emerging-only
  shared features: 66 | developed only (not used): driver_age, annual_mileage, telematics_opt_in, telematics_score, credit_band, telematics_score_known, annual_mileage_missing, customer_age
  emerging only (used in the emerging correction): gender, region
  dropped as near-duplicates: customer_tenure
  developed GLM: 979,968 rows x 192 coefficients, 9 iterations, converged: True
  largest relativities: ncb_years=b0 = 1.97; prior_claims_5y=3 = 1.82; prior_claims_5y=other = 1.59; driver_age_band=17_24 = 1.58; prior_claims_5y=2 = 1.53; cover_type=third_party_fire_theft = 0.71

==> buildings
  region: only 0% of emerging values exist in developed -> emerging-only
  shared features: 54 | developed only (not used): policyholder_age, credit_band, customer_age
  emerging only (used in the emerging correction): gender, region
  dropped as near-duplicates: customer_tenure
  developed GLM: 707,726 rows x 151 coeff

In [29]:
# %% Phase 4 Cell 3: Validation on the latest emerging year ====================
# Fit years = every emerging year but the latest; holdout = the latest year.
# For each line:
#   1. Estimate tau^2 (empirical Bayes) from the fit years: how far emerging
#      estimates sit from developed ones, beyond what emerging noise explains.
#      Shared features and emerging-only features get separate tau^2.
#   2. Fit the emerging correction for each multiple of the SHARED features'
#      tau^2 in tau_grid and score the holdout year with the competition metric
#      (emerging-only features always use their own estimated tau^2).
#        multiple 0   = Option 2: developed relativities, emerging level only
#        multiple 1   = Option 4: Bayesian credibility with the estimated tau^2
#        multiple inf = Option 1: emerging data only
#   3. Record which multiple scores best (used by Cell 4 if tau_choice = "holdout").

def eb_tau2(b_new, b_prior, var):
    """Method-of-moments tau^2: average squared gap minus the part noise explains.
    Only reasonably estimated coefficients (standard error < 1) are used."""
    ok = var < 1
    if ok.sum() < 3:
        return 1e-4
    return max(1e-4, float(np.mean((b_new[ok] - b_prior[ok]) ** 2 - var[ok])))

def credibility_fit(fit_d, prior, label=""):
    """Everything needed to fit the emerging correction on fit_d."""
    Xs, _ = design(fit_d, prior["spec"])
    new_spec, _ = make_spec(fit_d, prior["em_only"], min_level_expo_new, min_level_claims_new)
    Xn, new_names = design(fit_d, new_spec)
    X = np.hstack([Xs, Xn])
    y, expo = fit_d["target"].to_numpy(float), fit_d["exposure"].to_numpy(float)
    ks = Xs.shape[1]
    # emerging-only fit (weak prior for stability) -> estimates and their noise
    raw = fit_tweedie(X, y, np.log(expo), prior_sd=weak_prior_sd)
    var_e = np.diag(raw["cov"])[1:]
    var_d = np.diag(prior["cov"])[1:]
    tau_s = eb_tau2(raw["beta"][1:ks + 1], prior["beta"][1:], var_e[:ks] + var_d)
    tau_n = eb_tau2(raw["beta"][ks + 1:], np.zeros(X.shape[1] - ks), var_e[ks:])
    offset = np.log(expo) + prior["beta"][0] + lin_pred(Xs, prior["beta"])   # developed prediction
    own = np.array([n.split("=")[0] in own_level_features for n in prior["names"]] + [False] * Xn.shape[1])
    return {"X": X, "y": y, "offset": offset, "ks": ks, "new_spec": new_spec, "new_names": new_names, "own": own,
            "tau_s": tau_s, "tau_n": tau_n, "phi": raw["phi"], "var_e": var_e, "raw_beta": raw["beta"]}

def penalties(cf, mult, own_on=False, mult_new=1.0):
    def lam(tau):
        if mult == 0: return 1e12
        if np.isinf(mult): return cf["phi"] / weak_prior_sd ** 2          # emerging only (weak prior)
        return cf["phi"] / (mult * tau)
    k = cf["X"].shape[1]
    # mult scales the SHARED features' prior; mult_new scales the emerging-only features' prior
    # (region, gender: no developed estimate, so 0 means "no effect")
    lam_new = 1e12 if mult_new == 0 else cf["phi"] / (mult_new * cf["tau_n"])
    pen = np.concatenate([np.full(cf["ks"], lam(cf["tau_s"])), np.full(k - cf["ks"], lam_new)])
    if own_on:
        pen[cf["own"]] = cf["phi"] / weak_prior_sd ** 2     # own-level features: emerging's own estimate
    return pen

def predict_rows(d, prior, new_spec, corr_beta):
    Xs, _ = design(d, prior["spec"]); Xn, _ = design(d, new_spec)
    X = np.hstack([Xs, Xn])
    eta = (np.log(d["exposure"].to_numpy(float)) + prior["beta"][0] + lin_pred(Xs, prior["beta"])   # developed prediction
           + corr_beta[0] + lin_pred(X, corr_beta))                                                 # emerging correction
    return np.exp(eta)

mult_name = lambda m: "0 (developed relativities)" if m == 0 else ("inf (emerging only)" if np.isinf(m) else f"{m:g}")
val_rows, best, ho_preds, tau_info = [], {}, {}, []
for l in lines:
    print(f"\n==> {l}")
    with open(os.path.join(model_dir, f"prior_{l}.pkl"), "rb") as fh:
        prior = pickle.load(fh)
    em, _ = load_line(target_market, l)
    ho_year = em["policy_year"].max()
    fit_d, ho_d = em[em["policy_year"] < ho_year], em[em["policy_year"] == ho_year]
    if fit_d.empty:
        raise RuntimeError(f"{l}: only one emerging year - nothing to validate on")
    cf = credibility_fit(fit_d, prior)
    print(f"  estimated tau^2: shared features {cf['tau_s']:.4f} (prior sd {np.sqrt(cf['tau_s']):.3f} on the log scale),"
          f" emerging-only features {cf['tau_n']:.4f} | emerging-only features used: {', '.join(cf['new_spec']) or 'none'}")
    if cf["tau_s"] <= 1e-4:
        print("  -> no measurable difference between the markets' shared effects beyond emerging noise: they borrow almost fully")
    if cf["own"].any():
        print(f"  own-level features (emerging estimate, not shrunk): {', '.join(own_level_features)}")
    tau_info.append(dict(line=l, tau2_shared=cf["tau_s"], tau2_emerging_only=cf["tau_n"], phi=cf["phi"]))
    actual, expo, load = ho_d["target_uncapped"].to_numpy(float), ho_d["exposure"].to_numpy(float), ho_d["large_loading"].iloc[0]
    for own_on in ([False, True] if cf["own"].any() else [False]):
        for mult_new in (tau_grid_new if len(cf["new_names"]) else [1.0]):
            for mult in tau_grid:
                fit = fit_tweedie(cf["X"], cf["y"], cf["offset"], penalty=penalties(cf, mult, own_on, mult_new))
                pred = predict_rows(ho_d, prior, cf["new_spec"], fit["beta"]) * load
                val_rows.append(dict(line=l, own_level=own_on, new_feature_multiple=mult_new, tau2_multiple=mult_name(mult), mult=mult,
                                     d2=d2(actual, pred, expo), a_over_e=actual.sum() / pred.sum(),
                                     c_star=c_star(actual, pred), converged=fit["converged"]))
                ho_preds[(l, (mult, own_on, mult_new))] = pred
    v = pd.DataFrame([r for r in val_rows if r["line"] == l])
    top = v.loc[v["d2"].idxmax()]
    best[l] = (float(top["mult"]), bool(top["own_level"]), float(top["new_feature_multiple"]))
    ho_preds[(l, "actual")], ho_preds[(l, "expo")] = actual, expo
    ho_preds[(l, "nb")] = ho_d["new_business"].astype(str).str.lower().to_numpy() if "new_business" in ho_d else None
    del em, fit_d, ho_d, cf; gc.collect()

val = pd.DataFrame(val_rows)
print("\nHoldout results (latest emerging year): best 5 settings per line - higher D^2 is better")
show(val.sort_values(["line", "d2"], ascending=[True, False]).groupby("line").head(5).drop(columns="mult"),
     dec={"d2": 4, "a_over_e": 3, "c_star": 3})
print("\nEffect of each setting on its own (best D^2 per line when the other settings are free)")
for col in ["tau2_multiple", "new_feature_multiple", "own_level"]:
    print(f"  {col}:"); print(val.groupby(["line", col])["d2"].max().unstack(col).round(4).to_string())

# All lines together, as the competition scores them (one file)
def combined(choice):
    a = np.concatenate([ho_preds[(l, "actual")] for l in lines])
    p = np.concatenate([ho_preds[(l, choice[l])] for l in lines])
    e = np.concatenate([ho_preds[(l, "expo")] for l in lines])
    return d2(a, p, e)
desc = lambda c: f"shared x{mult_name(c[0])}, region/gender x{c[2]:g}{' + own new-business level' if c[1] else ''}"
summary = [dict(choice="best per line: " + "; ".join(f"{l}: {desc(best[l])}" for l in lines), d2_all_lines=combined(best))]
for mn in tau_grid_new:
    for mult in (0, 1, float("inf")):
        key = {l: (mult, False, mn) for l in lines}
        if all((l, key[l]) in ho_preds for l in lines):
            summary.append(dict(choice=f"{desc((mult, False, mn))} for every line", d2_all_lines=combined(key)))
summary = pd.DataFrame(summary)
print("\nAll lines together on the holdout year"); show(summary, dec={"d2_all_lines": 4})

# Lift check for the chosen model: actual should rise with predicted
lift = []
for l in lines:
    p, a, e = ho_preds[(l, best[l])], ho_preds[(l, "actual")], ho_preds[(l, "expo")]
    dec = pd.qcut(pd.Series(p / e).rank(method="first"), 5, labels=False) + 1
    g = pd.DataFrame({"q": dec, "a": a, "p": p, "e": e}).groupby("q").sum()
    lift.append(pd.DataFrame({"line": l, "quintile": g.index, "actual_rate": g["a"] / g["e"], "predicted_rate": g["p"] / g["e"]}))
lift = pd.concat(lift, ignore_index=True)
print("\nLift on the holdout year by predicted-rate quintile (chosen multiple)")
show(lift, dec={"actual_rate": 1, "predicted_rate": 1})

# New business is 76% of the 2025 book, so its level matters most: check it separately
nb_rows = []
for l in lines:
    nb = ho_preds[(l, "nb")]
    if nb is None: continue
    for v in sorted(set(nb)):
        sel = nb == v
        a, p = ho_preds[(l, "actual")][sel], ho_preds[(l, best[l])][sel]
        nb_rows.append(dict(line=l, new_business=v, rows=int(sel.sum()), actual=a.sum(), predicted=p.sum(), a_over_e=a.sum() / p.sum()))
if nb_rows:
    print("\nHoldout actual vs expected by new business (chosen multiple) - the 2025 book is mostly new business")
    show(pd.DataFrame(nb_rows), money=["actual", "predicted"], dec={"a_over_e": 3})

val.to_csv(os.path.join(model_dir, "validation_holdout.csv"), index=False)
summary.to_csv(os.path.join(model_dir, "validation_all_lines.csv"), index=False)
pd.DataFrame(tau_info).to_csv(os.path.join(model_dir, "validation_tau2.csv"), index=False)
with open(os.path.join(model_dir, "validation_choice.json"), "w") as fh:
    json.dump({"best_multiple": {l: (None if np.isinf(m) else m) for l, (m, o, n) in best.items()},
               "best_is_inf": {l: bool(np.isinf(m)) for l, (m, o, n) in best.items()},
               "own_level": {l: o for l, (m, o, n) in best.items()},
               "new_feature_multiple": {l: n for l, (m, o, n) in best.items()}}, fh)


==> motor
  estimated tau^2: shared features 0.0001 (prior sd 0.010 on the log scale), emerging-only features 0.0089 | emerging-only features used: gender, region
  -> no measurable difference between the markets' shared effects beyond emerging noise: they borrow almost fully
  own-level features (emerging estimate, not shrunk): new_business

==> buildings
  estimated tau^2: shared features 0.0001 (prior sd 0.010 on the log scale), emerging-only features 0.1623 | emerging-only features used: gender, region
  -> no measurable difference between the markets' shared effects beyond emerging noise: they borrow almost fully
  own-level features (emerging estimate, not shrunk): new_business

==> contents
  estimated tau^2: shared features 0.0001 (prior sd 0.010 on the log scale), emerging-only features 0.0001 | emerging-only features used: gender, region
  -> no measurable difference between the markets' shared effects beyond emerging noise: they borrow almost fully
  own-level features (eme

In [30]:
# %% Phase 4 Cell 4: Final fit on all emerging years, predict, submit ==========
# Re-estimates tau^2 on all emerging years, fits the correction with the chosen
# multiple, predicts every emerging test row (rate x exposure x large-claim
# loading), shows the credibility table, checks the file and writes submission.csv.

with open(os.path.join(model_dir, "validation_choice.json")) as fh:
    vc = json.load(fh)
chosen = {l: (float("inf") if vc["best_is_inf"][l] else vc["best_multiple"][l]) if tau_choice == "holdout" else 1.0
          for l in lines}
own_chosen = {l: bool(vc.get("own_level", {}).get(l, False)) if tau_choice == "holdout" else False for l in lines}
new_chosen = {l: float(vc.get("new_feature_multiple", {}).get(l, 1.0)) if tau_choice == "holdout" else 1.0 for l in lines}
print("Used per line:", "; ".join(f"{l}: shared tau^2 x {mult_name(m)}, region/gender tau^2 x {new_chosen[l]:g}"
                                  f"{' + own new-business level' if own_chosen[l] else ''}" for l, m in chosen.items()))

preds, cred_tables, fit_info, gate_flags = [], [], [], []
for l in lines:
    print(f"\n==> {l}")
    with open(os.path.join(model_dir, f"prior_{l}.pkl"), "rb") as fh:
        prior = pickle.load(fh)
    em, em_test = load_line(target_market, l)
    cf = credibility_fit(em, prior)
    fit = fit_tweedie(cf["X"], cf["y"], cf["offset"], penalty=penalties(cf, chosen[l], own_chosen[l], new_chosen[l]), label="emerging correction")
    load = em["large_loading"].iloc[0]
    p = predict_rows(em_test, prior, cf["new_spec"], fit["beta"]) * load
    if not np.isfinite(p).all() or (p < 0).any():
        raise RuntimeError(f"{l}: invalid predictions")

    # Credibility table: developed effect, emerging correction, Z and final effect per coefficient
    ks = cf["ks"]; corr = fit["beta"][1:]
    mult = chosen[l]
    tau_sh = np.full(ks, np.inf if np.isinf(mult) else mult * cf["tau_s"])
    tau = np.concatenate([tau_sh, np.full(len(corr) - ks, new_chosen[l] * cf["tau_n"])])   # emerging-only: own tau^2
    Z = np.where(np.isinf(tau), 1.0, tau / (tau + cf["var_e"]))
    if own_chosen[l]:
        Z = np.where(cf["own"], 1.0, Z)                     # own-level features use emerging's estimate
    dev_b = np.concatenate([prior["beta"][1:], np.zeros(len(corr) - ks)])
    ct = pd.DataFrame({"line": l, "term": prior["names"] + cf["new_names"],
                       "source": ["shared"] * ks + ["emerging only"] * (len(corr) - ks),
                       "developed_relativity": np.exp(dev_b), "emerging_alone_relativity": np.exp(cf["raw_beta"][1:]),
                       "Z": Z, "final_relativity": np.exp(dev_b + corr)})
    cred_tables.append(ct)
    flagged = ct[(ct["final_relativity"] > max_relativity) | (ct["final_relativity"] < 1 / max_relativity)]
    if len(flagged):
        gate_flags.append(f"{l}: {', '.join(flagged['term'])}")

    train_rate = em["target_uncapped"].sum() / em["exposure"].sum()
    train_pred = predict_rows(em, prior, cf["new_spec"], fit["beta"]) * load
    fit_info.append(dict(line=l, train_rows=len(em), test_rows=len(em_test), coefficients=len(corr),
                         tau2_multiple=mult_name(mult), own_new_business_level=own_chosen[l], region_gender_multiple=new_chosen[l], level_vs_developed=float(np.exp(fit["beta"][0])),
                         converged=fit["converged"], train_actual_rate=train_rate,
                         train_pred_rate=train_pred.sum() / em["exposure"].sum(),
                         test_pred_rate=p.sum() / em_test["exposure"].sum(),
                         test_new_business_share=float(em_test["new_business"].astype(str).str.lower().eq("true").mean())
                                                 if "new_business" in em_test else np.nan))
    preds.append(pd.DataFrame({"line": l, "orig_row_id": em_test["orig_row_id"].values, "prediction": p}))
    del em, em_test, cf; gc.collect()

fit_info = pd.DataFrame(fit_info)
cred = pd.concat(cred_tables, ignore_index=True)
print("\nFinal fits: level_vs_developed = emerging price level relative to the developed model;")
print("train_pred_rate should be close to train_actual_rate; test_pred_rate differs only through the 2025 mix (e.g. new business)")
show(fit_info, dec={"level_vs_developed": 3, "train_actual_rate": 1, "train_pred_rate": 1, "test_pred_rate": 1, "test_new_business_share": 3})
print("\nCredibility by feature (average Z across its levels; 1 = emerging data, 0 = developed)")
cred["feature"] = cred["term"].str.split("=").str[0]
show(cred.groupby(["line", "source", "feature"]).agg(levels=("Z", "size"), avg_Z=("Z", "mean"),
                                                     max_final_relativity=("final_relativity", "max"),
                                                     min_final_relativity=("final_relativity", "min")).reset_index()
     .sort_values(["line", "source", "avg_Z"], ascending=[True, False, False]),
     dec={"avg_Z": 3, "max_final_relativity": 3, "min_final_relativity": 3})
cred.to_csv(os.path.join(model_dir, "credibility_table.csv"), index=False)
fit_info.to_csv(os.path.join(model_dir, "final_fit_info.csv"), index=False)

# ---- Review gate -------------------------------------------------------------
if gate_flags:
    print(f"\nREVIEW: final relativities beyond {max_relativity}x:", "; ".join(gate_flags))
    if not reviewed_ok:
        raise RuntimeError("Review the flagged relativities; if acceptable set reviewed_ok = True in Cell 1 and rerun")

# ---- Submission ----------------------------------------------------------------
raw = [os.path.dirname(p) for r in ["/kaggle/input", "data"]
       for p in glob.glob(os.path.join(r, "**", "sample_submission.*"), recursive=True)
       if os.path.basename(os.path.dirname(p)) == target_market]
if len(raw) != 1:
    raise RuntimeError(f"Found {len(raw)} sample_submission files for {target_market}: {raw}")
ss_path = glob.glob(os.path.join(raw[0], "sample_submission.*"))[0]
sample_sub = pd.read_parquet(ss_path) if ss_path.endswith(".parquet") else pd.read_csv(ss_path, dtype={"row_id": str})
sample_sub["row_id"] = sample_sub["row_id"].astype(str)

allp = pd.concat(preds, ignore_index=True)
sub = allp.groupby("orig_row_id", as_index=False)["prediction"].sum().rename(columns={"orig_row_id": "row_id"})
sub = sample_sub[["row_id"]].merge(sub, on="row_id", how="left")
if len(sub) != len(sample_sub):              raise RuntimeError("Row count differs from sample_submission")
if sub["prediction"].isna().any():           raise RuntimeError(f"{int(sub['prediction'].isna().sum())} rows without a prediction")
if (sub["prediction"] < 0).any():            raise RuntimeError("Negative predictions")
if set(sub["row_id"]) != set(sample_sub["row_id"]): raise RuntimeError("row_ids differ from sample_submission")

sub.to_csv(os.path.join(work_dir, "submission.csv"), index=False)
allp.to_parquet(os.path.join(model_dir, "glm_test_predictions.parquet"), index=False)   # kept for the boosting step
print(f"\nSaved submission.csv: {len(sub):,} rows, mean prediction {sub['prediction'].mean():.2f}")
print(allp.groupby("line")["prediction"].agg(["count", "mean"]).round(2).to_string())

Used per line: motor: shared tau^2 x 0 (developed relativities), region/gender tau^2 x 4; buildings: shared tau^2 x 4, region/gender tau^2 x 0 + own new-business level; contents: shared tau^2 x 1, region/gender tau^2 x 4

==> motor
  emerging correction: 47,283 rows x 198 coefficients, 3 iterations, converged: True

==> buildings
  emerging correction: 27,447 rows x 157 coefficients, 4 iterations, converged: True

==> contents
  emerging correction: 34,566 rows x 159 coefficients, 4 iterations, converged: True

Final fits: level_vs_developed = emerging price level relative to the developed model;
train_pred_rate should be close to train_actual_rate; test_pred_rate differs only through the 2025 mix (e.g. new business)
     line  train_rows  test_rows  coefficients              tau2_multiple  own_new_business_level  region_gender_multiple  level_vs_developed  converged  train_actual_rate  train_pred_rate  test_pred_rate  test_new_business_share
    motor       47283      90521           

In [31]:
# =============================================================================
# PART 2 - PHASE 4b: BOOSTING CORRECTION ON TOP OF THE CREDIBILITY GLM
#
# The GLM prices each factor on its own; it can't see combinations such as
# weather x property type or young driver x new business. A LightGBM model is
# trained to learn ONLY what the GLM gets wrong: it starts from the GLM's
# prediction (init_score = log of the GLM prediction) and adds small
# multiplicative corrections, with the same Tweedie (power 1.5) loss as the metric.
#
#   - Trained on developed rows (GLM = the developed prior) plus emerging rows
#     (GLM = the credibility model), with a market flag, so interactions are
#     learned mostly from the large market and can still differ by market.
#   - Shallow trees, a minimum of rows per leaf and early stopping on rows the
#     trees never saw keep it from learning noise.
#   - Validation: the same emerging holdout year as Phase 4. Per line, the GLM,
#     the boosted model and a 50/50 blend are compared; the best is kept, so the
#     boosting is used only where it beats the GLM.
#   - Level kept from the GLM: Tweedie boosting on sparse claims drifts the
#     overall level (leaves of claim-free rows pull down faster than the few
#     claim rows pull up). So the boosted predictions are rescaled to the GLM's
#     total on the emerging training rows: the boosting re-ranks risks, the
#     validated GLM sets the level. The rescaling factor is printed.
#   - The GLM stays the explainable core: the boosting is a bounded correction.
#
# Needs Phase 4 Cells 1 and 2 to have run (functions and the developed prior).
# =============================================================================

# %% Phase 4b Cell 1: Settings and helpers =====================================

# ---- SETTINGS ---------------------------------------------------------------
gbm_params = dict(objective="tweedie", tweedie_variance_power=1.5,   # = the metric
                  learning_rate=0.02, num_leaves=7, max_depth=3,   # small trees: simple interactions only
                  min_data_in_leaf=2000,            # each leaf needs many rows: no small noisy groups
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l2=50.0,
                  cat_smooth=50, cat_l2=10, min_data_per_group=500, max_cat_threshold=16,   # restrain many-level categories
                  verbose=-1, seed=0, num_threads=4)
max_rounds        = 1000
early_stop_rounds = 100
stop_share        = 0.15      # share of training POLICIES set aside to decide when to stop adding trees
emerging_weight   = 3.0       # emerging rows count this many times in training (they are the market we predict)
blend_weights     = [0.0, 0.25, 0.5, 1.0]   # share of the boosted model in the final blend, tried on the holdout
max_gbm_levels    = 20        # categories with more levels than this are left to the GLM (trees can memorise them)
guard_level, guard_corr = 0.10, 0.80   # final check: boosted 2025 level within 10% of the GLM, rankings correlated
# -----------------------------------------------------------------------------

import lightgbm as lgb

if "fit_tweedie" not in globals() or not os.path.exists(os.path.join(model_dir, "prior_motor.pkl")):
    raise RuntimeError("Run Phase 4 Cells 1 and 2 first (functions and the developed prior)")

# ---- Credibility GLM helpers (identical to Phase 4 Cell 3, so this phase runs on its own)
def eb_tau2(b_new, b_prior, var):
    """Method-of-moments tau^2: average squared gap minus the part noise explains.
    Only reasonably estimated coefficients (standard error < 1) are used."""
    ok = var < 1
    if ok.sum() < 3:
        return 1e-4
    return max(1e-4, float(np.mean((b_new[ok] - b_prior[ok]) ** 2 - var[ok])))

def credibility_fit(fit_d, prior, label=""):
    """Everything needed to fit the emerging correction on fit_d."""
    Xs, _ = design(fit_d, prior["spec"])
    new_spec, _ = make_spec(fit_d, prior["em_only"], min_level_expo_new, min_level_claims_new)
    Xn, new_names = design(fit_d, new_spec)
    X = np.hstack([Xs, Xn])
    y, expo = fit_d["target"].to_numpy(float), fit_d["exposure"].to_numpy(float)
    ks = Xs.shape[1]
    # emerging-only fit (weak prior for stability) -> estimates and their noise
    raw = fit_tweedie(X, y, np.log(expo), prior_sd=weak_prior_sd)
    var_e = np.diag(raw["cov"])[1:]
    var_d = np.diag(prior["cov"])[1:]
    tau_s = eb_tau2(raw["beta"][1:ks + 1], prior["beta"][1:], var_e[:ks] + var_d)
    tau_n = eb_tau2(raw["beta"][ks + 1:], np.zeros(X.shape[1] - ks), var_e[ks:])
    offset = np.log(expo) + prior["beta"][0] + lin_pred(Xs, prior["beta"])   # developed prediction
    own = np.array([n.split("=")[0] in own_level_features for n in prior["names"]] + [False] * Xn.shape[1])
    return {"X": X, "y": y, "offset": offset, "ks": ks, "new_spec": new_spec, "new_names": new_names, "own": own,
            "tau_s": tau_s, "tau_n": tau_n, "phi": raw["phi"], "var_e": var_e, "raw_beta": raw["beta"]}

def penalties(cf, mult, own_on=False, mult_new=1.0):
    def lam(tau):
        if mult == 0: return 1e12
        if np.isinf(mult): return cf["phi"] / weak_prior_sd ** 2          # emerging only (weak prior)
        return cf["phi"] / (mult * tau)
    k = cf["X"].shape[1]
    # mult scales the SHARED features' prior; mult_new scales the emerging-only features' prior
    # (region, gender: no developed estimate, so 0 means "no effect")
    lam_new = 1e12 if mult_new == 0 else cf["phi"] / (mult_new * cf["tau_n"])
    pen = np.concatenate([np.full(cf["ks"], lam(cf["tau_s"])), np.full(k - cf["ks"], lam_new)])
    if own_on:
        pen[cf["own"]] = cf["phi"] / weak_prior_sd ** 2     # own-level features: emerging's own estimate
    return pen

def predict_rows(d, prior, new_spec, corr_beta):
    Xs, _ = design(d, prior["spec"]); Xn, _ = design(d, new_spec)
    X = np.hstack([Xs, Xn])
    eta = (np.log(d["exposure"].to_numpy(float)) + prior["beta"][0] + lin_pred(Xs, prior["beta"])   # developed prediction
           + corr_beta[0] + lin_pred(X, corr_beta))                                                 # emerging correction
    return np.exp(eta)

mult_name = lambda m: "0 (developed relativities)" if m == 0 else ("inf (emerging only)" if np.isinf(m) else f"{m:g}")

with open(os.path.join(model_dir, "validation_choice.json")) as fh:
    _vc = json.load(fh)
glm_choice = {l: (float("inf") if _vc["best_is_inf"][l] else _vc["best_multiple"][l],
                  bool(_vc.get("own_level", {}).get(l, False)),
                  float(_vc.get("new_feature_multiple", {}).get(l, 1.0))) for l in lines}

def load_prior(l):
    with open(os.path.join(model_dir, f"prior_{l}.pkl"), "rb") as fh:
        return pickle.load(fh)

def glm_emerging(fit_d, prior, l):
    """Credibility GLM fitted on fit_d with the settings Phase 4 chose; returns a predictor."""
    cf = credibility_fit(fit_d, prior)
    m, own_on, mn = glm_choice[l]
    fit = fit_tweedie(cf["X"], cf["y"], cf["offset"], penalty=penalties(cf, m, own_on, mn))
    return lambda d: predict_rows(d, prior, cf["new_spec"], fit["beta"])

def glm_developed(d, prior):
    Xs, _ = design(d, prior["spec"])
    return np.exp(np.log(d["exposure"].to_numpy(float)) + prior["beta"][0] + lin_pred(Xs, prior["beta"]))

def gbm_frame(d, feats, is_emerging, cats):
    """Features for the boosting model: raw columns (money relative to their market
    median), categories as LightGBM categories with the same codes everywhere."""
    d = money_relative(d)
    x = pd.DataFrame({c: (d[c] if c in d else pd.Series(np.nan, index=d.index)) for c in feats})   # emerging-only columns are blank for developed rows
    for c in feats:
        if c in cats:
            x[c] = pd.Categorical(x[c].where(x[c].isna(), x[c].astype(str)), categories=cats[c])
        else:
            x[c] = x[c].astype(float)
    x["is_emerging"] = float(is_emerging)
    return x

def category_levels(frames, feats):
    cats = {}
    for c in feats:
        cols = [f[c] for f in frames if c in f]
        if any(not pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s) for s in cols):
            cats[c] = sorted(set().union(*[set(s.astype(str).unique()) for s in cols]))
    return cats

def train_gbm(X, y, base, weight, groups, rounds=None):
    """Fit the correction. If rounds is None, stop early on a set-aside share of policies."""
    init = np.log(np.maximum(base, 1e-10))
    if rounds is None:
        rng = np.random.default_rng(0)
        pol = pd.Series(groups).astype(str)
        held = pol.isin(set(rng.choice(pol.unique(), int(stop_share * pol.nunique()), replace=False)))
        tr, va = ~held.to_numpy(), held.to_numpy()
        dtr = lgb.Dataset(X[tr], y[tr], weight=weight[tr], init_score=init[tr], free_raw_data=False)
        dva = lgb.Dataset(X[va], y[va], weight=weight[va], init_score=init[va], reference=dtr)
        bst = lgb.train(gbm_params, dtr, max_rounds, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(early_stop_rounds, verbose=False)])
        return bst, bst.best_iteration
    dall = lgb.Dataset(X, y, weight=weight, init_score=init)
    return lgb.train(gbm_params, dall, rounds), rounds

def gbm_predict(bst, X, base):
    return np.exp(np.log(np.maximum(base, 1e-10)) + bst.predict(X, raw_score=True))

def level_factor(bst, X_em, base_em):
    """Rescale so the boosted total on emerging training rows equals the GLM total."""
    return base_em.sum() / gbm_predict(bst, X_em, base_em).sum()

def gbm_features(dev, em, prior):
    """Features both markets have plus emerging-only ones (region, gender); developed-only
    features are left out (emerging rows would have nothing to follow)."""
    f_dev, f_em = feature_cols(dev), feature_cols(em)
    feats = [f for f in f_em if f in f_dev] + [f for f in f_em if f not in f_dev]
    many = [f for f in feats if (not pd.api.types.is_numeric_dtype(em[f]) or pd.api.types.is_bool_dtype(em[f]))
            and pd.concat([em[f].astype(str), dev[f].astype(str) if f in dev else em[f].astype(str)]).nunique() > max_gbm_levels]
    if many:
        print(f"  left to the GLM (more than {max_gbm_levels} levels): {', '.join(many)}")
    return [f for f in feats if f not in many]

print("Phase 4b setup done. GLM settings from Phase 4:",
      "; ".join(f"{l}: shared x{mult_name(m)}, region/gender x{n:g}{' + own NB' if o else ''}" for l, (m, o, n) in glm_choice.items()))

Phase 4b setup done. GLM settings from Phase 4: motor: shared x0 (developed relativities), region/gender x4; buildings: shared x4, region/gender x0 + own NB; contents: shared x1, region/gender x4


In [32]:
# %% Phase 4b Cell 2: Validation - does boosting beat the GLM? =================
# Same holdout as Phase 4: emerging fit years -> latest emerging year.
# Training rows: all developed rows + emerging fit-year rows. Holdout rows are
# never used for training or for deciding when to stop.

gbm_val, gbm_choice, gbm_rounds, hold = [], {}, {}, {}
for l in lines:
    print(f"\n==> {l}")
    prior = load_prior(l)
    dev, _ = load_line(source_market, l)
    em, _  = load_line(target_market, l)
    ho_year = em["policy_year"].max()
    fit_e, ho_e = em[em["policy_year"] < ho_year], em[em["policy_year"] == ho_year]
    feats = gbm_features(dev, em, prior)
    cats = category_levels([dev, em], feats)

    glm_e = glm_emerging(fit_e, prior, l)                       # credibility GLM on fit years only
    base_dev, base_fit, base_ho = glm_developed(dev, prior), glm_e(fit_e), glm_e(ho_e)
    X = pd.concat([gbm_frame(dev, feats, 0, cats), gbm_frame(fit_e, feats, 1, cats)], ignore_index=True)
    y = np.concatenate([dev["target"].to_numpy(float), fit_e["target"].to_numpy(float)])
    base = np.concatenate([base_dev, base_fit])
    w = np.concatenate([np.ones(len(dev)), np.full(len(fit_e), emerging_weight)])
    groups = np.concatenate([("d" + dev["orig_row_id"].astype(str)).to_numpy(), ("e" + fit_e["orig_row_id"].astype(str)).to_numpy()])
    if "policy_id" in dev and "policy_id" in fit_e:
        groups = np.concatenate([("d" + dev["policy_id"].astype(str)).to_numpy(), ("e" + fit_e["policy_id"].astype(str)).to_numpy()])
    bst, n_rounds = train_gbm(X, y, base, w, groups)
    gbm_rounds[l] = n_rounds
    print(f"  boosting: {len(X):,} training rows ({len(fit_e):,} emerging), {len(feats)} features, stopped at {n_rounds} trees")

    k = level_factor(bst, gbm_frame(fit_e, feats, 1, cats), base_fit)
    print(f"  level drift of the boosting on emerging training rows: x{1 / k:.3f} -> rescaled by {k:.3f}")
    load = ho_e["large_loading"].iloc[0]
    p_glm = base_ho * load
    p_gbm = gbm_predict(bst, gbm_frame(ho_e, feats, 1, cats), base_ho) * k * load
    actual, expo = ho_e["target_uncapped"].to_numpy(float), ho_e["exposure"].to_numpy(float)
    for bw in blend_weights:
        p = (1 - bw) * p_glm + bw * p_gbm
        gbm_val.append(dict(line=l, boosted_share=bw, d2=d2(actual, p, expo), a_over_e=actual.sum() / p.sum(), c_star=c_star(actual, p)))
        hold[(l, bw)] = p
    hold[(l, "actual")], hold[(l, "expo")] = actual, expo
    v = pd.DataFrame([r for r in gbm_val if r["line"] == l])
    gbm_choice[l] = float(v.loc[v["d2"].idxmax(), "boosted_share"])

    imp = pd.Series(bst.feature_importance("gain"), index=X.columns).sort_values(ascending=False)
    print("  what the boosting corrects most (share of its gain):",
          "; ".join(f"{k} {v:.0%}" for k, v in (imp / imp.sum()).head(8).items()))
    del dev, em, fit_e, ho_e, X, bst; gc.collect()

gbm_val = pd.DataFrame(gbm_val)
print("\nHoldout D^2 by share of the boosted model (0 = GLM only, 1 = boosted only)")
show(gbm_val, dec={"d2": 4, "a_over_e": 3, "c_star": 3})

def combined_gbm(choice):
    a = np.concatenate([hold[(l, "actual")] for l in lines]); e = np.concatenate([hold[(l, "expo")] for l in lines])
    return d2(a, np.concatenate([hold[(l, choice[l])] for l in lines]), e)
summ = pd.DataFrame([dict(choice="GLM only (Phase 4)", d2_all_lines=combined_gbm({l: 0.0 for l in lines})),
                     dict(choice="50/50 blend every line", d2_all_lines=combined_gbm({l: 0.5 for l in lines})),
                     dict(choice="boosted every line", d2_all_lines=combined_gbm({l: 1.0 for l in lines})),
                     dict(choice="best per line: " + ", ".join(f"{l}={gbm_choice[l]:g}" for l in lines),
                          d2_all_lines=combined_gbm(gbm_choice))])
print("\nAll lines together on the holdout year"); show(summ, dec={"d2_all_lines": 4})
gbm_val.to_csv(os.path.join(model_dir, "gbm_validation.csv"), index=False)
with open(os.path.join(model_dir, "gbm_choice.json"), "w") as fh:
    json.dump({"boosted_share": gbm_choice, "rounds": gbm_rounds}, fh)


==> motor
  left to the GLM (more than 20 levels): make_model, occupation, previous_insurer
  boosting: 1,001,648 training rows (21,680 emerging), 65 features, stopped at 452 trees
  level drift of the boosting on emerging training rows: x0.974 -> rescaled by 1.027
  what the boosting corrects most (share of its gain): region 17%; written_premium 11%; years_licensed 11%; rainfall_mm 7%; ncb_years 5%; mean_temperature_c 3%; years_at_address 3%; vehicle_colour 3%

==> buildings
  left to the GLM (more than 20 levels): occupation, previous_insurer
  boosting: 720,306 training rows (12,580 emerging), 54 features, stopped at 103 trees
  level drift of the boosting on emerging training rows: x0.908 -> rescaled by 1.101
  what the boosting corrects most (share of its gain): region 20%; elevation_m 19%; mean_temperature_c 8%; property_age 8%; sum_insured 5%; hail_days 4%; wildfire_index 4%; written_premium 4%

==> contents
  left to the GLM (more than 20 levels): occupation, previous_insurer


In [33]:
# %% Phase 4b Cell 3: Final fit, predict and submit =============================
# Credibility GLM on all emerging years (as Phase 4 Cell 4), boosting correction
# on all developed + all emerging rows with the number of trees validation found
# (scaled up for the extra data), blended per line as validation chose.
# Writes submission.csv (the final blend) and keeps the GLM-only file as
# submission_glm_only.csv for comparison.

with open(os.path.join(model_dir, "gbm_choice.json")) as fh:
    gc_ = json.load(fh)
share, rounds = gc_["boosted_share"], gc_["rounds"]
print("Boosted share per line:", ", ".join(f"{l} = {share[l]:g}" for l in lines))

final_preds, info = [], []
for l in lines:
    print(f"\n==> {l}")
    prior = load_prior(l)
    dev, _ = load_line(source_market, l)
    em, em_test = load_line(target_market, l)
    glm_e = glm_emerging(em, prior, l)
    base_test = glm_e(em_test)
    load = em["large_loading"].iloc[0]
    p_glm = base_test * load
    if share[l] > 0:
        feats = gbm_features(dev, em, prior)
        cats = category_levels([dev, em, em_test], feats)
        X = pd.concat([gbm_frame(dev, feats, 0, cats), gbm_frame(em, feats, 1, cats)], ignore_index=True)
        y = np.concatenate([dev["target"].to_numpy(float), em["target"].to_numpy(float)])
        base = np.concatenate([glm_developed(dev, prior), glm_e(em)])
        w = np.concatenate([np.ones(len(dev)), np.full(len(em), emerging_weight)])
        n_final = int(round(rounds[l] / (1 - stop_share)))
        bst, _ = train_gbm(X, y, base, w, None, rounds=n_final)
        base_em = base[len(dev):]
        k = level_factor(bst, gbm_frame(em, feats, 1, cats), base_em)
        p_gbm = gbm_predict(bst, gbm_frame(em_test, feats, 1, cats), base_test) * k * load
        print(f"  boosting: {n_final} trees on {len(X):,} rows; level rescaled by {k:.3f} to the GLM")
        del X, bst
    else:
        p_gbm = p_glm
        print("  boosting not used for this line (the GLM scored better on the holdout)")
    p = (1 - share[l]) * p_glm + share[l] * p_gbm
    if not np.isfinite(p).all() or (p < 0).any():
        raise RuntimeError(f"{l}: invalid predictions")
    # Guard: the boosting may re-rank risks but must not move the 2025 level or scramble the ranking
    if share[l] > 0:
        lvl = p.sum() / p_glm.sum() - 1
        cor = float(np.corrcoef(np.log(p_glm), np.log(p))[0, 1])
        if abs(lvl) > guard_level or cor < guard_corr:
            print(f"  GUARD: boosted predictions moved the level by {lvl:+.1%} or correlate only {cor:.2f} with the GLM"
                  f" -> GLM used for {l}")
            p = p_glm; share[l] = 0.0
    info.append(dict(line=l, boosted_share=share[l], glm_rate=p_glm.sum() / em_test["exposure"].sum(),
                     final_rate=p.sum() / em_test["exposure"].sum(),
                     corr_glm_final=float(np.corrcoef(np.log(p_glm), np.log(p))[0, 1]) if share[l] > 0 else 1.0))
    final_preds.append(pd.DataFrame({"line": l, "orig_row_id": em_test["orig_row_id"].values, "glm": p_glm, "prediction": p}))
    del dev, em, em_test; gc.collect()

info = pd.DataFrame(info)
print("\nPredicted cost per policy-year on the 2025 rows: GLM vs final (the boosting should refine, not shift, the level)")
show(info, dec={"glm_rate": 1, "final_rate": 1, "corr_glm_final": 3})

raw = [os.path.dirname(p) for r in ["/kaggle/input", "data"]
       for p in glob.glob(os.path.join(r, "**", "sample_submission.*"), recursive=True)
       if os.path.basename(os.path.dirname(p)) == target_market]
if len(raw) != 1:
    raise RuntimeError(f"Found {len(raw)} sample_submission files for {target_market}: {raw}")
ss_path = glob.glob(os.path.join(raw[0], "sample_submission.*"))[0]
sample_sub = pd.read_parquet(ss_path) if ss_path.endswith(".parquet") else pd.read_csv(ss_path, dtype={"row_id": str})
sample_sub["row_id"] = sample_sub["row_id"].astype(str)

allp = pd.concat(final_preds, ignore_index=True)
for col, fname in [("prediction", "submission.csv"), ("glm", "submission_glm_only.csv")]:
    sub = allp.groupby("orig_row_id", as_index=False)[col].sum().rename(columns={"orig_row_id": "row_id", col: "prediction"})
    sub = sample_sub[["row_id"]].merge(sub, on="row_id", how="left")
    if len(sub) != len(sample_sub) or sub["prediction"].isna().any() or (sub["prediction"] < 0).any() \
            or set(sub["row_id"]) != set(sample_sub["row_id"]):
        raise RuntimeError(f"{fname}: failed the format checks")
    sub.to_csv(os.path.join(work_dir, fname), index=False)
    print(f"Saved {fname}: {len(sub):,} rows, mean prediction {sub['prediction'].mean():.2f}")
allp.to_parquet(os.path.join(model_dir, "final_test_predictions.parquet"), index=False)

Boosted share per line: motor = 1, buildings = 0.5, contents = 0.5

==> motor
  left to the GLM (more than 20 levels): make_model, occupation, previous_insurer
  boosting: 532 trees on 1,027,251 rows; level rescaled by 1.025 to the GLM

==> buildings
  left to the GLM (more than 20 levels): occupation, previous_insurer
  boosting: 121 trees on 735,173 rows; level rescaled by 1.056 to the GLM

==> contents
  left to the GLM (more than 20 levels): occupation, previous_insurer
  boosting: 207 trees on 756,183 rows; level rescaled by 1.038 to the GLM

Predicted cost per policy-year on the 2025 rows: GLM vs final (the boosting should refine, not shift, the level)
     line  boosted_share  glm_rate  final_rate  corr_glm_final
    motor            1.0     544.7       535.2           0.991
buildings            0.5     465.0       478.3           0.994
 contents            0.5      99.5        99.9           0.996
Saved submission.csv: 209,063 rows, mean prediction 338.84
Saved submission_glm_o

In [34]:
# %% Phase 5: Office premium build-up for the reference policies ===============
# The brief asks, for each of the 20 reference policies, how the office premium
# (the premium charged) is built from our predicted risk premium with the
# expenses table. All rates are read from expenses.parquet (emerging column);
# nothing is typed in except the assumptions listed below.
#
#   office premium = (risk premium + cat reinsurance + fixed expense)
#                    / (1 - variable expense - commission - premium tax - profit margin)
#
# Assumptions (state these in the report):
#   - Items expressed as a share of premium are shares of the OFFICE premium,
#     so they sit in the denominator (the premium must pay for them itself).
#   - Cat reinsurance = its rate x risk premium, for property lines only.
#   - Fixed expense is per policy per year, so a part-year row pays its share
#     (x exposure); set fixed_pro_rata = False to charge the full amount per row.
#   - A blank emerging rate (comparison-site commission) uses the developed rate.

# ---- SETTINGS ----------------------------------------------------------------
target_market  = "emerging"
prediction_file = "submission.csv"          # the final Week 2 submission
property_lines = ["buildings", "contents"]  # lines that carry cat reinsurance
fixed_pro_rata = True
# -----------------------------------------------------------------------------

import os, glob
import numpy as np, pandas as pd
work_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
folder = [os.path.dirname(p) for r in ["/kaggle/input", "data"]
          for p in glob.glob(os.path.join(r, "**", "expenses.parquet"), recursive=True)
          if os.path.basename(os.path.dirname(p)) == target_market]
if len(folder) != 1:
    raise RuntimeError(f"Found {len(folder)} expenses files for {target_market}: {folder}")
exp = pd.read_parquet(os.path.join(folder[0], "expenses.parquet"))
ref = pd.read_parquet(os.path.join(folder[0], "reference_policies.parquet"))
ref["row_id"] = ref["row_id"].astype(str)
pred = pd.read_csv(os.path.join(work_dir, prediction_file), dtype={"row_id": str})

rate, notes = {}, []
for _, r in exp.iterrows():
    v = r[target_market]
    if pd.isna(v):
        v = r["developed"]
        notes.append(f"{r['item']}: no {target_market} value -> developed value {v:g} used")
    rate[r["item"]] = float(v)

ref = ref.merge(pred, on="row_id", how="left")
if ref["prediction"].isna().any():
    raise RuntimeError("Some reference policies have no prediction in " + prediction_file)
line = ref["line"].astype(str).str.lower()
chan = ref["channel"].astype(str).str.lower()
out = pd.DataFrame({"row_id": ref["row_id"], "line": line, "channel": chan, "exposure": ref["exposure"].round(4),
                    "new_business": ref["new_business"]})
out["risk_premium"]   = ref["prediction"]
out["cat_reins"]      = np.where(line.isin(property_lines), rate["cat_reinsurance"] * out["risk_premium"], 0.0)
out["fixed_expense"]  = rate["fixed_expense_per_policy"] * (out["exposure"] if fixed_pro_rata else 1.0)
commission            = chan.map(lambda c: rate.get(f"commission_{c}", np.nan))
if commission.isna().any():
    raise RuntimeError(f"No commission rate for channel(s): {sorted(chan[commission.isna()].unique())}")
out["loading_share"]  = rate["variable_expense"] + commission + rate["premium_tax"] + rate["target_profit_margin"]
out["office_premium"] = (out["risk_premium"] + out["cat_reins"] + out["fixed_expense"]) / (1 - out["loading_share"])
op = out["office_premium"]
out["variable_exp"]   = rate["variable_expense"] * op
out["commission"]     = commission * op
out["premium_tax"]    = rate["premium_tax"] * op
out["profit"]         = rate["target_profit_margin"] * op
check = out[["risk_premium", "cat_reins", "fixed_expense", "variable_exp", "commission", "premium_tax", "profit"]].sum(axis=1)
assert np.allclose(check, op), "components don't add up to the office premium"
out["risk_share_of_op"] = out["risk_premium"] / op

print("Rates used (" + target_market + "):", "; ".join(f"{k} = {v:g}" for k, v in rate.items()))
for n in notes: print("Note:", n)
cols = ["row_id", "line", "channel", "exposure", "risk_premium", "cat_reins", "fixed_expense", "variable_exp",
        "commission", "premium_tax", "profit", "office_premium", "risk_share_of_op"]
print(out[cols].round({"risk_premium": 2, "cat_reins": 2, "fixed_expense": 2, "variable_exp": 2, "commission": 2,
                       "premium_tax": 2, "profit": 2, "office_premium": 2, "risk_share_of_op": 3}).to_string(index=False))
out[cols].to_csv(os.path.join(work_dir, "office_premium_reference_policies.csv"), index=False)
print("\nSaved office_premium_reference_policies.csv - every row: components add up to the office premium")

Rates used (emerging): fixed_expense_per_policy = 85; variable_expense = 0.15; commission_direct = 0; commission_comparison_site = 0.1; commission_broker = 0.2; premium_tax = 0.08; cat_reinsurance = 0.12; target_profit_margin = 0.08
Note: commission_comparison_site: no emerging value -> developed value 0.1 used
   row_id      line channel  exposure  risk_premium  cat_reins  fixed_expense  variable_exp  commission  premium_tax  profit  office_premium  risk_share_of_op
M00161473     motor  broker    0.9644        928.15       0.00          81.97        309.22      412.30       164.92  164.92         2061.49             0.450
M00176777     motor  broker    0.9918        915.87       0.00          84.30        306.18      408.23       163.29  163.29         2041.17             0.449
M00185311     motor  direct    0.9644        767.72       0.00          81.97        184.72        0.00        98.51   98.51         1231.43             0.623
M00167496     motor  broker    0.9836        468.42

In [35]:
# %% Phase 6: Sensitivity and scenario table (READ-ONLY) =========================
# How much would the 2025 predicted cost per policy-year move if a key assumption
# were different? One assumption is changed at a time, starting from the final
# predictions. This cell only READS saved files and writes model/sensitivity_table.csv;
# it does not touch submission.csv or any model.
#
# Stresses (each shown as the % change in the 2025 predicted cost per policy-year):
#   1. Trend: emerging-observed trend; precision-weighted blend of emerging and
#      developed trends; developed trend +/- 2 points a year.
#   2. Year level: price as if 2025 resembled only the earlier or only the latest
#      emerging year (the buildings 2023 vs 2024 gap).
#   3. Large claims: cap at other quantiles, and loading fully emerging or fully developed.
#   4. New business: its predicted cost x 0.90 and x 1.10.
#   5. Office premium only: comparison-site commission 5% / 15% instead of 10%.
# Approximation: the model level moves with the training target's level, so a
# change to the target scales the predictions by the same ratio (exact for the
# level, ignores small changes in relativities).

# ---- SETTINGS ----------------------------------------------------------------
source_market, target_market = "developed", "emerging"
lines          = ["motor", "buildings", "contents"]
trend_power    = 1.5
trend_shift    = 0.02                    # +/- 2 points a year
cap_quantiles  = [0.99, 0.995, 0.9975]   # 0.995 = used in the model
nb_factors     = [0.90, 1.10]
commission_alt = [0.05, 0.15]            # comparison-site commission alternatives (10% assumed)
# -----------------------------------------------------------------------------

import os, glob
import numpy as np, pandas as pd
work_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T = lambda m: os.path.join(work_dir, f"target_{m}")
preds = pd.read_parquet(os.path.join(work_dir, "model", "final_test_predictions.parquet"))

def trend_row(m, l):
    t = pd.read_csv(os.path.join(T(m), "step7_trend.csv"))
    r = t[(t["line"] == l) & (t["power"] == trend_power) & (t["mix_adjusted"])].iloc[0]
    se = (np.log1p(r["upper"]) - np.log1p(r["lower"])) / (2 * 1.96)
    return float(r["trend"]), float(se)

rows = []
def add(l, group, scenario, factor, note=""):
    rows.append(dict(line=l, stress=group, scenario=scenario, change=factor - 1, note=note))

for l in lines:
    tgt  = pd.read_parquet(os.path.join(T(target_market), f"target_final_{l}.parquet"))
    s7   = pd.read_parquet(os.path.join(T(target_market), f"target_rows_{l}_step7.parquet"))
    test = pd.read_parquet(os.path.join(work_dir, f"clean_{target_market}", f"test_{l}.parquet"),
                           columns=["orig_row_id", "exposure", "new_business", "channel"])
    p = preds[preds["line"] == l]
    pred_year = int(pd.read_csv(os.path.join(T(target_market), "step7_trend_factors.csv"))["pred_year"].iloc[0])

    # ---- 1. Trend ----------------------------------------------------------
    applied = pd.read_csv(os.path.join(T(target_market), "step7_applied.csv")).set_index("line").loc[l, "applied"]
    t_em, se_em = trend_row(target_market, l)
    t_dev, se_dev = trend_row(source_market, l)
    z = se_dev ** 2 / (se_dev ** 2 + se_em ** 2)            # weight on emerging: its precision share
    t_blend = np.expm1(z * np.log1p(t_em) + (1 - z) * np.log1p(applied))
    base = s7.groupby("policy_year")["cost_target"].sum()
    level = lambda t: sum(base[y] * (1 + t) ** (pred_year - y) for y in base.index)
    central = level(applied)
    add(l, "trend", f"emerging observed ({t_em:+.1%}/yr)", level(t_em) / central, "noisy: 95% interval shown in Phase 3")
    add(l, "trend", f"precision-weighted blend ({t_blend:+.1%}/yr, weight on emerging {z:.2f})", level(t_blend) / central)
    add(l, "trend", f"developed {applied:+.1%} {'+' if trend_shift > 0 else ''}{trend_shift:.0%}/yr", level(applied + trend_shift) / central)
    add(l, "trend", f"developed {applied:+.1%} -{trend_shift:.0%}/yr", level(applied - trend_shift) / central)

    # ---- 2. Year level -------------------------------------------------------
    by_year = tgt.groupby("policy_year").agg(c=("target_uncapped", "sum"), e=("exposure", "sum"))
    all_rate = by_year["c"].sum() / by_year["e"].sum()
    for y, r in by_year.iterrows():
        add(l, "year level", f"2025 like {y} only", (r["c"] / r["e"]) / all_rate, "range if one year's experience repeats")

    # ---- 3. Large claims -----------------------------------------------------
    s8 = pd.read_csv(os.path.join(T(target_market), "step8_large_claims.csv")).set_index("line").loc[l]
    dev7 = pd.read_parquet(os.path.join(T(source_market), f"target_rows_{l}_step7.parquet"), columns=["cost_trended"])["cost_trended"]
    em7 = s7["cost_trended"]
    Z = float(s8["Z"])
    def level_at(q, z_override=None):
        out = {}
        for name, c in (("em", em7), ("dev", dev7)):
            pos = c[c > 0]; cap = np.quantile(pos, q)
            out[name] = c.sum() / np.minimum(c, cap).sum()
        zz = Z if z_override is None else z_override
        loading = zz * out["em"] + (1 - zz) * out["dev"]
        return np.minimum(em7, np.quantile(em7[em7 > 0], q)).sum() * loading
    central_l = level_at(0.995)
    for q in cap_quantiles:
        if q != 0.995:
            add(l, "large claims", f"cap at {q:.2%} quantile", level_at(q) / central_l)
    add(l, "large claims", "loading 100% emerging", level_at(0.995, 1.0) / central_l, f"model uses Z = {Z:.2f}")
    add(l, "large claims", "loading 100% developed", level_at(0.995, 0.0) / central_l)

    # ---- 4. New business -------------------------------------------------------
    nb = test.drop_duplicates("orig_row_id").set_index("orig_row_id")["new_business"].astype(str).str.lower()
    is_nb = p["orig_row_id"].map(nb).eq("true").to_numpy()
    tot = p["prediction"].sum()
    for f in nb_factors:
        note = "x0.90 scored 0.14785 vs 0.15288 on the public board (motor)" if (f == 0.90 and l == "motor") else ""
        add(l, "new business", f"new business x{f:.2f}", (tot + (f - 1) * p["prediction"].to_numpy()[is_nb].sum()) / tot, note)

    # ---- 5. Comparison-site commission (office premium, not risk premium) -----
    ch = test.drop_duplicates("orig_row_id").set_index("orig_row_id")["channel"].astype(str).str.lower()
    share_cs = p["orig_row_id"].map(ch).eq("comparison_site").mean()
    other = 0.15 + 0.08 + 0.08                      # variable + tax + profit (emerging expenses)
    for c in commission_alt:
        f = (1 - other - 0.10) / (1 - other - c)
        add(l, "office premium", f"comparison-site commission {c:.0%} (not 10%)",
            1 + share_cs * (f - 1), f"{share_cs:.1%} of policies; their office premium changes {f - 1:+.1%}")

    rate = tot / test["exposure"].sum()
    rows.append(dict(line=l, stress="CENTRAL", scenario=f"final model: {rate:,.1f} per policy-year", change=0.0, note=""))

sens = pd.DataFrame(rows)
sens["rate_2025"] = [np.nan] * len(sens)
for l in lines:
    c = sens[(sens["line"] == l) & (sens["stress"] == "CENTRAL")]["scenario"].iloc[0]
    rate = float(c.split(": ")[1].split(" ")[0].replace(",", ""))
    sens.loc[sens["line"] == l, "rate_2025"] = rate * (1 + sens.loc[sens["line"] == l, "change"])
sens["change"] = sens["change"].map(lambda x: f"{x:+.1%}")
pd.set_option("display.width", 250, "display.max_colwidth", 90)
order = {"CENTRAL": 0, "trend": 1, "year level": 2, "large claims": 3, "new business": 4, "office premium": 5}
sens = sens.sort_values(["line", "stress"], key=lambda s: s.map(order) if s.name == "stress" else s).reset_index(drop=True)
print(sens[["line", "stress", "scenario", "change", "rate_2025", "note"]].round({"rate_2025": 1}).to_string(index=False))

# Envelope: the widest single-assumption range per line (stresses are not additive)
env = []
for l in lines:
    s = sens[(sens["line"] == l) & ~sens["stress"].isin(["CENTRAL", "office premium"])
             & ~sens["scenario"].str.startswith("emerging observed")]   # rejected as noise in Phase 3
    env.append(dict(line=l, central=sens[(sens["line"] == l) & (sens["stress"] == "CENTRAL")]["rate_2025"].iloc[0],
                    low=s["rate_2025"].min(), high=s["rate_2025"].max(),
                    widest_driver=s.loc[(s["rate_2025"] - s["rate_2025"].median()).abs().idxmax(), "scenario"]))
env = pd.DataFrame(env)
print("\nLow / central / high 2025 cost per policy-year (widest single stress; the emerging-observed trend is excluded, rejected as noise in Phase 3)")
print(env.round(1).to_string(index=False))
sens.to_csv(os.path.join(work_dir, "model", "sensitivity_table.csv"), index=False)
env.to_csv(os.path.join(work_dir, "model", "sensitivity_envelope.csv"), index=False)
print("\nSaved model/sensitivity_table.csv and model/sensitivity_envelope.csv (submission.csv not touched)")

     line         stress                                                     scenario change  rate_2025                                                        note
buildings        CENTRAL                           final model: 478.3 per policy-year  +0.0%      478.3                                                            
buildings          trend                                emerging observed (-33.5%/yr) -53.5%      222.6                        noisy: 95% interval shown in Phase 3
buildings          trend precision-weighted blend (+8.3%/yr, weight on emerging 0.01)  -0.5%      475.7                                                            
buildings          trend                                       developed +8.7% +2%/yr  +3.0%      492.6                                                            
buildings          trend                                       developed +8.7% -2%/yr  -3.0%      464.2                                                            
buildings     ye